# FBNetGen v5 — Best-Fold Selection + Node Importance (Colab)

This notebook does two things:

1. **Aggregates the 25 folds** in `results/advanced/fbnetgen_v5_full/` and picks the best model (highest subject-level test Pearson r).
2. **Runs node (ROI) importance** on that best fold using four complementary methods:
   * **Pool-gate attention** — the model's own attention pooling weights.
   * **Vanilla saliency** — `|d ŷ / d x_node|` summed across feature dims.
   * **Integrated Gradients** — Sundararajan et al. 2017, baseline = zeros.
   * **Occlusion** — zero-out one ROI at a time and measure prediction change.

All four methods are aggregated to a single ranked CSV. Top ROIs are visualised.

### Important notes about the checkpoints
The 25 fold checkpoints in `results/advanced/fbnetgen_v5_full/` come from **two different training scripts**:

| Folds | Script | Model class |
|---|---|---|
| `outer1_inner1..5` | `notebooks/colab_train_fbnetgen_v5_full.ipynb` | `models.fbnetgen.FBNetGenFromGraph` |
| `outer2..5_*` (20 folds) | `training/advanced/train_enhanced_models.py` | `models_enhanced.fbnetgen_enhanced.FBNetGenFromGraphEnhanced` |

The notebook auto-detects the architecture from the saved `state_dict` keys.

### Before running
1. **Runtime → Change runtime type → A100/L4 GPU** (T4 is fine too).
2. Ensure `folds_data/` and the `GNN-mri/` project are on Drive (same paths as the training notebook).
3. Ensure `results/advanced/fbnetgen_v5_full/` from your run is on Drive.
4. Edit the **CONFIG** cell, then run all cells.

## 1. Config

In [ ]:
DRIVE_FOLD_DIR    = '/content/drive/MyDrive/MRI_data'
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/GNN-mri'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/results/fbnetgen_v5_full'

# Output: where importance CSVs / plots will be saved
DRIVE_OUTPUT_DIR  = '/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability'

# Optional: path to a Shen 268 atlas CSV with columns like (node_index, region_name, network, hemisphere)
# Set to None if you don't have one — the notebook still works, just won't print region names.
ATLAS_CSV_PATH = '/content/GNN-mri/data/shen268_coords.csv'  # CSV with NodeNo, x_mni, y_mni, z_mni
AUTO_FETCH_SHEN_LABELS = True

# Which fold(s) to interpret. Accepts:
#   'best'                                  → single fold with highest subject-level test r
#   'all'                                   → every fold present and aggregate
#   'outer1', 'outer2', ... 'outer5'        → all 5 inner folds of one outer (handy when you only have outer-1)
#   'graphs_outer1_inner4'                  → one specific fold
#   ['graphs_outer1_inner1', ...]           → arbitrary list of folds to aggregate over
INTERPRET_TARGET = 'outer1'

# Number of top ROIs to display
TOP_K = 30

# Integrated Gradients steps (more = smoother but slower; 32–64 is plenty)
IG_STEPS = 32

# Batch size for inference (interpretability is per-window; AMP off here)
BATCH_SIZE = 64

# Whether to also run the (slow) per-ROI occlusion sweep
RUN_OCCLUSION = True

# Reuse existing interpretability outputs if they are already present in DRIVE_OUTPUT_DIR.
# This avoids recomputing slow XAI. Set False only when you intentionally want fresh node/edge importance.
USE_EXISTING_INTERPRETABILITY = True

# If True, writes into DRIVE_OUTPUT_DIR even when it already has files.
# Keep False for normal use; USE_EXISTING_INTERPRETABILITY already reads existing outputs.
OVERWRITE_INTERPRETABILITY_DIR = False


## 2. Mount Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

# ── Versioned output directory: never overwrite a prior run ──
# Strategy:
#   1. If the requested DRIVE_OUTPUT_DIR doesn't exist OR is empty, use it as-is.
#   2. Otherwise append _v2, _v3, ... up to _v99.
#   3. If even those collide (very unlikely), fall back to a timestamp suffix.
def _resolve_versioned_dir(base: Path) -> Path:
    base = Path(base)
    if not base.exists() or not any(base.iterdir()):
        return base
    for v in range(2, 100):
        cand = base.parent / f'{base.name}_v{v}'
        if not cand.exists() or not any(cand.iterdir()):
            return cand
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    return base.parent / f'{base.name}_{ts}'

requested_out = Path(DRIVE_OUTPUT_DIR)
if USE_EXISTING_INTERPRETABILITY or OVERWRITE_INTERPRETABILITY_DIR:
    out_dir = requested_out
else:
    out_dir = _resolve_versioned_dir(requested_out)
out_dir.mkdir(parents=True, exist_ok=True)

if out_dir == requested_out:
    mode = 'reuse existing' if USE_EXISTING_INTERPRETABILITY else ('overwrite allowed' if OVERWRITE_INTERPRETABILITY_DIR else 'fresh')
    print(f'Output dir: {out_dir}  ({mode})')
else:
    print(f'Requested  : {requested_out}  (already has files - not overwriting)')
    print(f'Output dir : {out_dir}  (new)')

import subprocess, torch

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])

torch_ver = torch.__version__.split('+')[0]
cuda_ver  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_ver}  |  CUDA build: {cuda_ver}')

pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html'
_run('pip install -q torch_geometric')
_run(f'pip install -q torch_scatter torch_sparse -f {pyg_url}')
_run('pip install -q scikit-learn scipy dill matplotlib seaborn openpyxl xlsxwriter')
print('Dependencies ready.')

## 3. Copy project + fold data to local disk

In [ ]:
import os, shutil, sys

PROJECT_ROOT = Path('/content/GNN-mri')
src = Path(DRIVE_PROJECT_DIR)
assert src.exists(), f'Project not found at {src}'

# Drive-safe copy: skip .gsheet / .gdoc / .gslides / .gform virtual files
# (these are Google Workspace pointers, not real files — FUSE returns ENOTSUP on read).
_drive_virtual = shutil.ignore_patterns(
    '*.gsheet', '*.gdoc', '*.gslides', '*.gform',
    '*.gmap', '*.gdraw', '*.gtable', '*.gsite',
    '.tmp.driveupload', '.tmp.drivedownload',
)
if not PROJECT_ROOT.exists():
    print('Copying project from Drive...')
    shutil.copytree(str(src), str(PROJECT_ROOT), ignore=_drive_virtual)
else:
    print('Project already at /content/GNN-mri')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOCAL_FOLD_DIR = Path('/content/folds_data')
DRIVE_FOLD_PATH = Path(DRIVE_FOLD_DIR)
assert DRIVE_FOLD_PATH.exists(), f'Fold data not found at {DRIVE_FOLD_PATH}'

if not LOCAL_FOLD_DIR.exists():
    print('Copying fold data to local disk...')
    shutil.copytree(str(DRIVE_FOLD_PATH), str(LOCAL_FOLD_DIR), ignore=_drive_virtual)
else:
    print('Fold data already on local disk.')

fold_files = sorted(LOCAL_FOLD_DIR.glob('graphs_outer*.pkl'))
print(f'Found {len(fold_files)} fold files.')

RESULTS_DIR = Path(DRIVE_RESULTS_DIR)
assert RESULTS_DIR.exists(), f'Results dir not found at {RESULTS_DIR}'
fold_dirs = sorted([d for d in RESULTS_DIR.iterdir() if d.is_dir() and d.name.startswith('graphs_outer')])
print(f'Found {len(fold_dirs)} fold result dirs in {RESULTS_DIR}')

## 4. Aggregate fold metrics — pick the best model

Each fold dir may contain:
* `fbnetgen_summary.json` (preferred — has `test_metrics.subject_level.pearson_r`)
* `fbnetgen_predictions.json` with embedded `test_metrics.subj_r` (older format)
* `fbnetgen_predictions.json` as a **list** of per-subject dicts (newer format) — already aggregated, recompute r locally
* `fbnetgen_best.pt` checkpoint with `val_metrics` and `test_metrics`

We try them in order so every fold gets a row.

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

rows = []
for fd in fold_dirs:
    fold_name = fd.name
    test_r = test_mse = val_r = np.nan
    src = ''

    # 1. Try fbnetgen_summary.json (newest format from train_enhanced_models.py)
    summary_p = fd / 'fbnetgen_summary.json'
    if summary_p.exists():
        with open(summary_p) as f:
            s = json.load(f)
        tm = s.get('test_metrics', {})
        sl = tm.get('subject_level', {})
        test_r   = sl.get('pearson_r', sl.get('r', np.nan))
        test_mse = sl.get('mse', np.nan)
        val_r    = s.get('best_validation', {}).get('subj_r', np.nan)
        src = 'summary'

    # 2. Try predictions.json with embedded test_metrics.subj_r (colab notebook format)
    if np.isnan(test_r):
        preds_p = fd / 'fbnetgen_predictions.json'
        if preds_p.exists():
            with open(preds_p) as f:
                p = json.load(f)
            if isinstance(p, dict) and 'test_metrics' in p:
                tm = p['test_metrics']
                test_r   = tm.get('subj_r', tm.get('r', np.nan))
                test_mse = tm.get('subj_mse', tm.get('mse', np.nan))
                src = 'preds_dict'
            elif isinstance(p, list) and p and 'subject_id' in p[0]:
                # Per-subject list — aggregate windows then compute r
                df = pd.DataFrame(p)
                # use _normalized columns if available (already on the same scale)
                if 'prediction_normalized' in df.columns:
                    pred_col, tgt_col = 'prediction_normalized', 'target_normalized'
                else:
                    pred_col, tgt_col = 'prediction', 'target'
                agg = df.groupby('subject_id').agg({pred_col: 'mean', tgt_col: 'first'}).reset_index()
                if len(agg) >= 2:
                    test_r, _ = pearsonr(agg[pred_col], agg[tgt_col])
                    test_mse = float(((agg[pred_col] - agg[tgt_col]) ** 2).mean())
                src = 'preds_list'

    # 3. Fall back to checkpoint val_metrics / test_metrics
    if np.isnan(val_r):
        ckpt_p = fd / 'fbnetgen_best.pt'
        if ckpt_p.exists():
            try:
                ck = torch.load(ckpt_p, map_location='cpu', weights_only=False)
                vm = ck.get('val_metrics', {})
                val_r = vm.get('subj_r', vm.get('r', val_r))
                if np.isnan(test_r):
                    tm = ck.get('test_metrics', {})
                    test_r   = tm.get('subj_r', tm.get('r', test_r))
                    test_mse = tm.get('subj_mse', tm.get('mse', test_mse))
                    if not src:
                        src = 'ckpt'
            except Exception as e:
                print(f'  could not load {ckpt_p}: {e}')

    rows.append({
        'fold': fold_name,
        'val_subj_r':  float(val_r) if val_r is not None else np.nan,
        'test_subj_r': float(test_r) if test_r is not None else np.nan,
        'test_subj_mse': float(test_mse) if test_mse is not None else np.nan,
        'source': src,
    })

summary_df = pd.DataFrame(rows).sort_values('test_subj_r', ascending=False).reset_index(drop=True)
print(summary_df.to_string(index=False))

summary_df.to_csv(out_dir / 'fold_metric_summary.csv', index=False)
print(f'\n→ Saved fold ranking to {out_dir / "fold_metric_summary.csv"}')
print(f'\nMean test_subj_r:  {summary_df["test_subj_r"].mean():.4f}  ± {summary_df["test_subj_r"].std():.4f}')
print(f'Median test_subj_r: {summary_df["test_subj_r"].median():.4f}')

best_row = summary_df.iloc[0]
print(f'\nBest fold: {best_row["fold"]}  →  test_subj_r = {best_row["test_subj_r"]:.4f}')

## 5. Architecture-aware checkpoint loader

Detects whether the saved `state_dict` is from `FBNetGenFromGraph` (basic) or `FBNetGenFromGraphEnhanced` and rebuilds the matching model.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

from models.fbnetgen import FBNetGenFromGraph
from models_enhanced.fbnetgen_enhanced import FBNetGenFromGraphEnhanced


def detect_arch(state_dict):
    """Return ('basic'|'enhanced', short reason)."""
    keys = set(state_dict.keys())
    # Enhanced trainer uses gnn_predictor.* and head.*; basic uses predictor.* and mlp.*
    if any(k.startswith('gnn_predictor.') for k in keys) and any(k.startswith('head.') for k in keys):
        return 'enhanced', 'has gnn_predictor.* and head.*'
    if any(k.startswith('predictor.') for k in keys) or any(k.startswith('node_encoder.') for k in keys):
        return 'basic', 'has predictor.* / node_encoder.*'
    raise RuntimeError(f'Cannot detect architecture from keys: {sorted(keys)[:6]} ...')


def load_fbnetgen(ckpt_path, in_dim=268, device='cuda'):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = ck.get('model_state_dict', ck.get('state_dict', ck))
    cfg = ck.get('config', {})
    arch, reason = detect_arch(sd)
    print(f'  arch = {arch}  ({reason})')
    if arch == 'basic':
        model = FBNetGenFromGraph(
            in_dim=in_dim,
            hidden_dim=cfg.get('hidden_dim', 128),
            n_layers=cfg.get('n_layers', 3),
            n_heads=cfg.get('n_heads', 2),
            dropout=cfg.get('dropout', 0.25),
            refine_graph=cfg.get('refine_graph', False),
        )
    else:
        # Enhanced: pull n_gnn_layers if present
        model = FBNetGenFromGraphEnhanced(
            in_dim=in_dim,
            hidden_dim=cfg.get('hidden_dim', 128),
            n_gnn_layers=cfg.get('n_gnn_layers', cfg.get('n_layers', 3)),
            n_heads=cfg.get('n_heads', 2),
            dropout=cfg.get('dropout', 0.25),
        )
    model.load_state_dict(sd, strict=True)
    model.to(device).eval()
    n_p = sum(p.numel() for p in model.parameters())
    print(f'  loaded {n_p:,} params')
    return model, cfg, arch

## 6. Interpretability methods (model-agnostic)

All four methods take a model + a PyG `Data`/`Batch` and return per-node importance shaped `(n_rois,)`.
We then average across all test windows for the chosen fold.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from utils.data_utils import load_graphs_with_normalization


def _forward_basic_with_gates(model, data):
    """Recompute FBNetGenFromGraph forward but expose pool_gate softmax weights per node."""
    x = model.node_encoder(data.x)
    edge_index = data.edge_index
    edge_attr = data.edge_attr if hasattr(data, 'edge_attr') else None
    if model.refine_graph and edge_index.size(1) > 0:
        src, dst = edge_index[0], edge_index[1]
        q = model.W_q(x[src]); k = model.W_k(x[dst])
        attn_score = torch.sigmoid(model.edge_scorer(q * k))
        ea = edge_attr if edge_attr is None or edge_attr.dim() == 2 else edge_attr.unsqueeze(-1)
        edge_weight = ea * attn_score if ea is not None else attn_score
    else:
        edge_weight = edge_attr if edge_attr is None or edge_attr.dim() == 2 else edge_attr.unsqueeze(-1)

    p = model.predictor
    h = p.input_proj(x)
    for i, (gat, norm) in enumerate(zip(p.gat_layers, p.norms)):
        h_res = h
        h = gat(h, edge_index, edge_attr=edge_weight)
        h = F.elu(h); h = norm(h)
        if i > 0:
            h = h + h_res
    gate_logits = p.pool_gate(h).squeeze(-1)
    return h, gate_logits


def _forward_enhanced_with_gates(model, data):
    x = model.encoder(data.x)
    edge_index = data.edge_index
    edge_attr = data.edge_attr if hasattr(data, 'edge_attr') else None
    if edge_index.size(1) > 0:
        src, dst = edge_index[0], edge_index[1]
        q = model.W_q(x[src]); k = model.W_k(x[dst])
        attn_score = torch.sigmoid(model.edge_scorer(q * k))
        ea = edge_attr if (edge_attr is None or edge_attr.dim() == 2) else edge_attr.unsqueeze(-1)
        edge_weight = ea * attn_score if ea is not None else attn_score
    else:
        edge_weight = edge_attr
    h = model.gnn_predictor(x, edge_index, edge_weight)
    gate_logits = model.pool_gate(h).squeeze(-1)
    return h, gate_logits


def collect_pool_gates(model, arch, loader, device, n_rois=268):
    """Per-node pool-gate softmax weights, accumulated to mean over all windows."""
    sums = torch.zeros(n_rois, device=device)
    counts = torch.zeros(n_rois, device=device)
    fwd = _forward_basic_with_gates if arch == 'basic' else _forward_enhanced_with_gates
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, gate_logits = fwd(model, batch)
            # per-graph softmax
            bs = int(batch.batch.max().item()) + 1
            for g in range(bs):
                mask = batch.batch == g
                w = torch.softmax(gate_logits[mask].float(), dim=0)
                # Each graph has exactly n_rois nodes (Shen 268), in canonical ROI order
                sums  += w
                counts += 1
    return (sums / counts.clamp(min=1)).cpu().numpy()


def compute_saliency(model, loader, device, n_rois=268):
    """|d ŷ / d x_node|, summed across feature dims, averaged across windows."""
    accum = torch.zeros(n_rois, device=device)
    n_graphs = 0
    model.eval()
    for batch in loader:
        batch = batch.to(device)
        x = batch.x.detach().clone().requires_grad_(True)
        # Wrap forward so model uses our x
        batch.x = x
        out = model(batch)
        out_sum = out.sum()
        grads = torch.autograd.grad(out_sum, x, retain_graph=False)[0]  # (N, F)
        sal = grads.abs().sum(dim=-1)  # (N,)
        bs = int(batch.batch.max().item()) + 1
        for g in range(bs):
            mask = batch.batch == g
            accum += sal[mask].detach()
            n_graphs += 1
    return (accum / max(n_graphs, 1)).cpu().numpy()


def compute_integrated_gradients(model, loader, device, n_rois=268, steps=32):
    """Integrated gradients with zero baseline. Per-window, then averaged."""
    accum = torch.zeros(n_rois, device=device)
    n_graphs = 0
    model.eval()
    for batch in loader:
        batch = batch.to(device)
        x_orig = batch.x.detach().clone()
        baseline = torch.zeros_like(x_orig)
        # Riemann midpoint approximation
        ig_grad = torch.zeros_like(x_orig)
        for s in range(steps):
            alpha = (s + 0.5) / steps
            x_interp = baseline + alpha * (x_orig - baseline)
            x_interp.requires_grad_(True)
            batch.x = x_interp
            out = model(batch)
            grads = torch.autograd.grad(out.sum(), x_interp, retain_graph=False)[0]
            ig_grad += grads.detach() / steps
        # Restore original x for downstream code
        batch.x = x_orig
        ig = (x_orig - baseline) * ig_grad      # (N, F)
        sal = ig.abs().sum(dim=-1)              # (N,)
        bs = int(batch.batch.max().item()) + 1
        for g in range(bs):
            mask = batch.batch == g
            accum += sal[mask]
            n_graphs += 1
    return (accum / max(n_graphs, 1)).cpu().numpy()


def compute_occlusion(model, loader, device, n_rois=268):
    """Per-ROI |Δŷ| when that ROI's feature row is zeroed. Averaged across windows.
    Note: this only zeros node features; edges stay intact (the LDW prior).
    """
    accum = torch.zeros(n_rois, device=device)
    n_graphs = 0
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            x_orig = batch.x.detach().clone()
            base_pred = model(batch).detach()                # (B,)
            bs = int(batch.batch.max().item()) + 1
            # For each ROI index 0..267, zero out that ROI in every graph in the batch and re-run
            for r in range(n_rois):
                x_mod = x_orig.clone()
                # Find global indices for ROI r in each graph (assumes canonical ordering: r-th node in each graph)
                idx = []
                for g in range(bs):
                    nodes_g = (batch.batch == g).nonzero(as_tuple=True)[0]
                    if r < nodes_g.numel():
                        idx.append(int(nodes_g[r].item()))
                if not idx:
                    continue
                x_mod[idx] = 0.0
                batch.x = x_mod
                pred = model(batch).detach()
                accum[r] += (pred - base_pred).abs().sum()
            batch.x = x_orig
            n_graphs += bs
    return (accum / max(n_graphs, 1)).cpu().numpy()


def occlusion_fast(model, loader, device, n_rois=268):
    """Vectorised occlusion: tile the batch n_rois times, zero a different ROI per copy.
    Memory-heavy; falls back to compute_occlusion on OOM.
    """
    return compute_occlusion(model, loader, device, n_rois=n_rois)

## 7. Run interpretability on the chosen fold

In [ ]:
import re

def _resolve_targets(spec, summary_df):
    """Turn INTERPRET_TARGET into a concrete list of fold dir names."""
    available = set(summary_df['fold'].tolist())

    if isinstance(spec, (list, tuple)):
        missing = [f for f in spec if f not in available]
        if missing:
            raise ValueError(f'Unknown folds: {missing}. Available: {sorted(available)[:5]} ...')
        return list(spec)

    if spec == 'best':
        # Best by test_subj_r; falls back to val_subj_r if test is NaN
        df = summary_df.copy()
        df['rank_key'] = df['test_subj_r'].fillna(df['val_subj_r'])
        return [df.sort_values('rank_key', ascending=False).iloc[0]['fold']]

    if spec == 'all':
        return summary_df['fold'].tolist()

    # 'outerN' — pick all 5 inner folds of one outer
    m = re.fullmatch(r'outer([1-9]\d*)', spec)
    if m:
        prefix = f'graphs_outer{m.group(1)}_inner'
        folds = sorted(f for f in available if f.startswith(prefix))
        if not folds:
            raise ValueError(f'No folds matching {prefix}* in {sorted(available)}')
        return folds

    # Single explicit fold name
    if spec in available:
        return [spec]

    raise ValueError(
        f'INTERPRET_TARGET={spec!r} not understood. '
        "Use 'best' | 'all' | 'outerN' | a fold name | a list of fold names."
    )


target_folds = _resolve_targets(INTERPRET_TARGET, summary_df)

print(f'INTERPRET_TARGET = {INTERPRET_TARGET!r}')
print(f'Running on {len(target_folds)} fold(s):')
for f in target_folds:
    row = summary_df[summary_df['fold'] == f].iloc[0]
    print(f'  • {f}   (test_subj_r={row["test_subj_r"]:.4f},  val_subj_r={row["val_subj_r"]:.4f})')

def _target_suffix(folds):
    return f'{"-".join(folds[:1])}{("_+" + str(len(folds)-1)) if len(folds) > 1 else ""}'

target_suffix = _target_suffix(target_folds)
print(f'Interpretability file suffix: {target_suffix}')



In [ ]:
import time
import numpy as np

def run_for_fold(fold_name):
    print(f'\n{"="*60}\n{fold_name}\n{"="*60}')
    ckpt_path = RESULTS_DIR / fold_name / 'fbnetgen_best.pt'
    fold_path = LOCAL_FOLD_DIR / f'{fold_name}.pkl'
    assert ckpt_path.exists(), f'Missing {ckpt_path}'
    assert fold_path.exists(), f'Missing fold pkl {fold_path}'

    # Load data first so we know in_dim
    train_g, val_g, test_g, info = load_graphs_with_normalization(
        str(fold_path), normalize_method='standard'
    )
    in_dim = train_g[0].x.size(-1)
    n_rois = train_g[0].x.size(0)
    print(f'  in_dim={in_dim}, n_rois={n_rois}, n_test_windows={len(test_g)}')

    model, cfg, arch = load_fbnetgen(ckpt_path, in_dim=in_dim, device=device)

    test_loader = DataLoader(test_g, batch_size=BATCH_SIZE, shuffle=False)

    t0 = time.time()
    pool_imp = collect_pool_gates(model, arch, test_loader, device, n_rois=n_rois)
    print(f'  pool-gate done   ({time.time()-t0:.1f}s)')

    t0 = time.time()
    sal_imp  = compute_saliency(model, test_loader, device, n_rois=n_rois)
    print(f'  saliency done    ({time.time()-t0:.1f}s)')

    t0 = time.time()
    ig_imp   = compute_integrated_gradients(model, test_loader, device, n_rois=n_rois, steps=IG_STEPS)
    print(f'  IG done          ({time.time()-t0:.1f}s,  steps={IG_STEPS})')

    if RUN_OCCLUSION:
        t0 = time.time()
        occ_imp = compute_occlusion(model, test_loader, device, n_rois=n_rois)
        print(f'  occlusion done   ({time.time()-t0:.1f}s)')
    else:
        occ_imp = np.full(n_rois, np.nan)

    df = pd.DataFrame({
        'roi_index': np.arange(n_rois),
        'pool_gate':       pool_imp,
        'saliency':        sal_imp,
        'integrated_grad': ig_imp,
        'occlusion':       occ_imp,
    })
    df['fold'] = fold_name
    df['arch'] = arch
    return df

saved_node_csv = out_dir / f'node_importance_{target_suffix}.csv'

if USE_EXISTING_INTERPRETABILITY and saved_node_csv.exists():
    print(f'\nLoading existing node importance: {saved_node_csv}')
    agg = pd.read_csv(saved_node_csv)
    all_imp = None
    print(f'Loaded {len(agg)} ROI rows; skipping slow node XAI recomputation.')
else:
    fold_dfs = []
    for fname in target_folds:
        fold_dfs.append(run_for_fold(fname))

    all_imp = pd.concat(fold_dfs, ignore_index=True)
    print(f'\nDone. Rows: {len(all_imp)} ({len(target_folds)} folds x n_rois)')


## 8. Aggregate, rank, save

We rank ROIs by each method (1 = most important), then average ranks across methods for a stable consensus.
When interpretability is run on multiple folds, we first average each method across folds before ranking.

In [ ]:
import io

method_cols = ['pool_gate', 'saliency', 'integrated_grad', 'occlusion']
all_imp = globals().get('all_imp', None)
if 'target_suffix' not in globals():
    node_candidates = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)
    if node_candidates:
        node_csv_inferred = node_candidates[-1]
        target_suffix = node_csv_inferred.stem.replace('node_importance_', '', 1)
        target_folds = globals().get('target_folds', [target_suffix.split('_+')[0]])
        print(f'Inferred target_suffix from saved CSV: {target_suffix}')
    elif 'target_folds' in globals():
        target_suffix = f'{"-".join(target_folds[:1])}{("_+" + str(len(target_folds)-1)) if len(target_folds) > 1 else ""}'
    else:
        target_suffix = str(globals().get('INTERPRET_TARGET', 'latest'))
        target_folds = [target_suffix]
if 'agg' not in globals() and all_imp is None:
    saved_node_csv = out_dir / f'node_importance_{target_suffix}.csv'
    if saved_node_csv.exists():
        print(f'Loading existing node importance: {saved_node_csv}')
        agg = pd.read_csv(saved_node_csv)
    else:
        csvs = sorted(out_dir.glob('node_importance_*.csv'))
        if not csvs:
            raise FileNotFoundError(f'No node_importance_*.csv found in {out_dir}')
        print(f'Loading existing node importance: {csvs[-1]}')
        agg = pd.read_csv(csvs[-1])

if 'agg' in globals() and all_imp is None:
    print('Using precomputed node importance table already loaded from CSV.')
else:
    agg = all_imp.groupby('roi_index')[method_cols].mean().reset_index()

    rank_cols = []
    for c in method_cols:
        if agg[c].isna().all():
            continue
        rcol = c + '_rank'
        agg[rcol] = agg[c].rank(ascending=False, method='min')
        rank_cols.append(rcol)

    agg['mean_rank'] = agg[rank_cols].mean(axis=1)
    agg = agg.sort_values('mean_rank').reset_index(drop=True)
    agg.insert(1, 'consensus_rank', np.arange(1, len(agg) + 1))

# Reuse path: infer/repair ranking columns when agg came from an existing CSV.
method_cols = [c for c in method_cols if c in agg.columns]
rank_cols = [c + '_rank' for c in method_cols if c + '_rank' in agg.columns]
if not rank_cols:
    for c in method_cols:
        if agg[c].isna().all():
            continue
        rcol = c + '_rank'
        agg[rcol] = agg[c].rank(ascending=False, method='min')
        rank_cols.append(rcol)
if rank_cols and 'mean_rank' not in agg.columns:
    agg['mean_rank'] = agg[rank_cols].mean(axis=1)
if 'consensus_rank' not in agg.columns and 'mean_rank' in agg.columns:
    agg = agg.sort_values('mean_rank').reset_index(drop=True)
    agg.insert(1, 'consensus_rank', np.arange(1, len(agg) + 1))
elif 'consensus_rank' in agg.columns:
    agg = agg.sort_values('consensus_rank').reset_index(drop=True)

def _coarse_hemi_lobe_from_xyz(x, y, z):
    hemi = 'Midline' if abs(float(x)) < 2 else ('Right' if float(x) > 0 else 'Left')
    if float(z) < -15:
        lobe = 'Cerebellum/BrainStem'
    elif float(y) > 20:
        lobe = 'Frontal'
    elif float(y) < -55:
        lobe = 'Occipital'
    elif float(z) > 45:
        lobe = 'Parietal'
    elif float(y) < -15:
        lobe = 'Temporal'
    else:
        lobe = 'Subcortical/Limbic'
    return hemi, lobe

def _read_shen_label_csv(source):
    if isinstance(source, Path):
        raw = source.read_text(encoding='utf-8-sig')
    elif isinstance(source, str):
        raw = source
        if '\n' not in source and '\r' not in source and len(source) < 512:
            try:
                p = Path(source)
                if p.exists():
                    raw = p.read_text(encoding='utf-8-sig')
            except (OSError, ValueError):
                pass
    else:
        raw = str(source)
    try:
        df = pd.read_csv(io.StringIO(raw))
        cols = {str(c).lower().strip() for c in df.columns}
        if len(df) == 268 or {'node', 'network'}.issubset(cols):
            return df
    except Exception:
        pass
    # The public canlab/NITRC network file can appear as: "Node,Network 1,2 2,4 ...".
    rows = []
    for token in raw.replace('Node,Network', '').replace('\n', ' ').split():
        token = token.strip().strip(',')
        if ',' not in token:
            continue
        a, b = token.split(',', 1)
        try:
            rows.append((int(a), int(b)))
        except ValueError:
            continue
    if len(rows) == 268:
        return pd.DataFrame(rows, columns=['Node', 'Network'])
    raise ValueError(f'Could not parse Shen label table; found {len(rows)} node/network rows')

def _standardize_roi_label_table(df, n_rois=268):
    df = df.copy()
    cols = {str(c).lower().strip(): c for c in df.columns}
    nodeno_col = next((cols[k] for k in ['nodeno', 'node_no', 'node no', 'node'] if k in cols), None)
    zero_idx_col = next((cols[k] for k in ['roi_index', 'node_index', 'roi', 'index'] if k in cols), None)
    if nodeno_col is not None:
        df['roi_index'] = df[nodeno_col].astype(int) - 1
    elif zero_idx_col is not None:
        df['roi_index'] = df[zero_idx_col].astype(int)
    elif len(df) == n_rois:
        df['roi_index'] = np.arange(n_rois)
    else:
        raise ValueError('Could not infer ROI index column from atlas label table')
    df = df[(df['roi_index'] >= 0) & (df['roi_index'] < n_rois)].copy()
    df = df.sort_values('roi_index').drop_duplicates('roi_index')
    rename_map = {}
    aliases = {
        'region_name': ['region_name', 'region', 'name', 'label', 'roi_name'],
        'network': ['network', 'networks', 'net_name', 'network_name'],
        'hemisphere': ['hemisphere', 'hemi'],
        'lobe': ['lobe'],
        'x_mni': ['x_mni', 'x'],
        'y_mni': ['y_mni', 'y'],
        'z_mni': ['z_mni', 'z'],
    }
    for canonical, names in aliases.items():
        for a in names:
            if a in cols:
                rename_map[cols[a]] = canonical
                break
    df = df.rename(columns=rename_map)
    keep = ['roi_index', 'region_name', 'network', 'hemisphere', 'lobe', 'x_mni', 'y_mni', 'z_mni']
    return df[[c for c in keep if c in df.columns]]

def build_shen268_label_table(n_rois=268):
    labels = pd.DataFrame({'roi_index': np.arange(n_rois)})
    labels['node_no'] = labels['roi_index'] + 1
    labels['roi_name'] = labels['node_no'].map(lambda x: f'Shen268 ROI {x}')
    candidates = []
    if ATLAS_CSV_PATH:
        candidates.append(Path(ATLAS_CSV_PATH))
    candidates += [
        Path('/content/drive/MyDrive/GNN-mri/data/shen_268_labels.csv'),
        Path('/content/drive/MyDrive/GNN-mri/data/shen268_coords.csv'),
        Path('/content/drive/MyDrive/GNN-mri/data/shen_268_coords.csv'),
        Path('/content/drive/MyDrive/GNN-mri/data/shen_268_parcellation_networklabels.csv'),
        Path('/content/GNN-mri/data/shen_268_labels.csv'),
        Path('/content/GNN-mri/data/shen268_coords.csv'),
        Path('/content/GNN-mri/data/shen_268_coords.csv'),
        Path('/content/GNN-mri/data/shen_268_parcellation_networklabels.csv'),
    ]
    for cp in candidates:
        try:
            if cp.exists():
                atlas = _standardize_roi_label_table(_read_shen_label_csv(cp), n_rois=n_rois)
                labels = labels.merge(atlas, on='roi_index', how='left')
                print(f'ROI labels joined from {cp}')
                break
        except Exception as e:
            print(f'Atlas candidate failed ({cp}): {e}')
    if AUTO_FETCH_SHEN_LABELS and ('network' not in labels.columns or labels['network'].isna().all()):
        try:
            import io, ssl, urllib.request
            url = ('https://raw.githubusercontent.com/canlab/Neuroimaging_Pattern_Masks/master/'
                   'Atlases_and_parcellations/2013_Shen_Constable_NIMG_268_parcellation/'
                   'shen_268_parcellation_networklabels.csv')
            ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
            raw = urllib.request.urlopen(url, timeout=15, context=ctx).read().decode('utf-8')
            net_df = _standardize_roi_label_table(_read_shen_label_csv(raw), n_rois=n_rois)
            net_names = {1: 'Medial Frontal', 2: 'Frontoparietal', 3: 'Default Mode', 4: 'Subcortical/Cerebellum', 5: 'Motor', 6: 'Visual I', 7: 'Visual II', 8: 'Visual Association'}
            if 'network' in net_df.columns:
                def _net_name(v):
                    try: return net_names.get(int(v), f'Network {int(v)}')
                    except Exception: return str(v)
                net_df['network'] = net_df['network'].map(_net_name)
                labels = labels.drop(columns=[c for c in ['network'] if c in labels.columns]).merge(net_df[['roi_index', 'network']], on='roi_index', how='left')
                print('ROI network labels fetched from canlab Shen-268 table')
        except Exception as e:
            print(f'Could not fetch Shen network labels ({type(e).__name__}: {e})')
    if {'x_mni', 'y_mni', 'z_mni'}.issubset(labels.columns):
        hemi_lobe = labels.apply(lambda r: _coarse_hemi_lobe_from_xyz(r['x_mni'], r['y_mni'], r['z_mni']), axis=1)
        auto_hemi = [h for h, _ in hemi_lobe]
        auto_lobe = [l for _, l in hemi_lobe]
        labels['hemisphere'] = labels.get('hemisphere', pd.Series(index=labels.index, dtype=object)).fillna(pd.Series(auto_hemi, index=labels.index))
        labels['lobe'] = labels.get('lobe', pd.Series(index=labels.index, dtype=object)).fillna(pd.Series(auto_lobe, index=labels.index))
    for c in ['region_name', 'network', 'hemisphere', 'lobe']:
        if c not in labels.columns:
            labels[c] = ''
        labels[c] = labels[c].fillna('').astype(str)
    labels['region_detail'] = (labels['hemisphere'] + ' ' + labels['lobe']).str.strip()
    labels['roi_label'] = labels.apply(lambda r: ' | '.join([x for x in [r['roi_name'], r['region_name'], r['region_detail'], r['network']] if str(x).strip()]), axis=1)
    return labels

roi_label_df = build_shen268_label_table(n_rois=len(agg))
label_cols = [c for c in ['roi_index', 'node_no', 'roi_name', 'roi_label', 'region_name', 'region_detail', 'network', 'hemisphere', 'lobe', 'x_mni', 'y_mni', 'z_mni'] if c in roi_label_df.columns]
for c in [c for c in label_cols if c != 'roi_index']:
    if c in agg.columns:
        agg = agg.drop(columns=[c])
agg = agg.merge(roi_label_df[label_cols], on='roi_index', how='left')

out_csv = out_dir / f'node_importance_{"-".join(target_folds[:1])}{("_+" + str(len(target_folds)-1)) if len(target_folds) > 1 else ""}.csv'
agg.to_csv(out_csv, index=False)
print(f'\n→ Saved per-ROI importance to {out_csv}')

print(f'\nTop {TOP_K} ROIs by consensus rank:')
show_cols = ['consensus_rank', 'roi_index', 'node_no', 'roi_label'] + method_cols + rank_cols
for opt in ['region_name', 'region_detail', 'network', 'hemisphere', 'lobe']:
    if opt in agg.columns and opt not in show_cols:
        show_cols.append(opt)
print(agg[show_cols].head(TOP_K).to_string(index=False))

## 9. Visualise

* Bar plot of top-K ROIs (consensus rank).
* Heat-map of all 4 methods × top-K ROIs.
* Spearman correlation between methods (sanity check).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

topk = agg.head(TOP_K).copy()

# 1. Bar plot — consensus importance (use mean of z-scored methods)
z_cols = []
for c in method_cols:
    if c not in agg.columns or agg[c].isna().all():
        continue
    zc = c + '_z'
    mu, sd = agg[c].mean(), agg[c].std() + 1e-12
    agg[zc] = (agg[c] - mu) / sd
    z_cols.append(zc)
agg['consensus_z'] = agg[z_cols].mean(axis=1)
topk_z = agg.sort_values('consensus_z', ascending=False).head(TOP_K)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(
    [str(int(r)) for r in topk_z['roi_index']][::-1],
    topk_z['consensus_z'][::-1],
    color='steelblue',
)
ax.set_xlabel('Consensus importance (mean of z-scored methods)')
ax.set_ylabel('ROI index')
ax.set_title(f'Top {TOP_K} ROIs — fold(s): {", ".join(target_folds)}')
plt.tight_layout()
plt.savefig(out_dir / 'top_rois_bar.png', dpi=150)
plt.show()

# 2. Heat-map of methods × top ROIs (z-scored)
import seaborn as sns
heat = topk_z.set_index('roi_index')[z_cols]
fig, ax = plt.subplots(figsize=(7, max(6, TOP_K * 0.3)))
sns.heatmap(heat, cmap='viridis', cbar_kws={'label': 'z-scored importance'}, ax=ax)
ax.set_title('Top ROIs × interpretability methods')
plt.tight_layout()
plt.savefig(out_dir / 'top_rois_heatmap.png', dpi=150)
plt.show()

# 3. Spearman between methods — agreement sanity check
available = [c for c in method_cols if c in agg.columns and not agg[c].isna().all()]
corr = pd.DataFrame(index=available, columns=available, dtype=float)
for a in available:
    for b in available:
        rho, _ = spearmanr(agg[a], agg[b])
        corr.loc[a, b] = rho
print('\nSpearman rank correlation between methods (full 268 ROIs):')
print(corr.astype(float).round(3))

## 10. Bundle everything into one Excel workbook

Easier to scroll than separate CSVs — one file with multiple sheets:

* `top_30` — the headline sheet, top-K consensus ROIs (sorted)
* `node_importance` — full 268-ROI table (sorted by consensus rank)
* `fold_summary` — per-fold val/test metrics
* `method_correlations` — Spearman matrix between methods
* `per_fold_long` — raw per-fold per-ROI scores (long form)
* `config` — what settings produced this run

In [ ]:
xlsx_path = out_dir / f'fbnetgen_interpretability_{"-".join(target_folds[:1])}{("_+" + str(len(target_folds)-1)) if len(target_folds) > 1 else ""}.xlsx'

# Build the headline top_K sheet — most-useful columns first
top_cols = ['consensus_rank', 'roi_index', 'node_no', 'roi_label', 'consensus_z'] + method_cols + rank_cols
for opt in ['region_name', 'region_detail', 'network', 'hemisphere', 'lobe', 'x_mni', 'y_mni', 'z_mni']:
    if opt in agg.columns and opt not in top_cols:
        top_cols.append(opt)
top_cols = [c for c in top_cols if c in agg.columns]
top_sheet = agg.head(TOP_K)[top_cols].copy()

# Full sheet with same columns
full_sheet = agg[top_cols].copy()

# Per-fold long-form table is only available when XAI was recomputed in this run.
per_fold = None
if 'all_imp' in globals() and all_imp is not None:
    per_fold = all_imp.sort_values(['fold', 'roi_index']).reset_index(drop=True)
    if 'roi_label_df' in globals():
        for c in [c for c in label_cols if c != 'roi_index']:
            if c in per_fold.columns:
                per_fold = per_fold.drop(columns=[c])
        per_fold = per_fold.merge(roi_label_df[label_cols], on='roi_index', how='left')
else:
    print('Skipping per_fold_long sheet because raw per-fold node scores were not recomputed in this session.')

# Spearman correlation between methods (recompute so we always have it)
from scipy.stats import spearmanr
avail = [c for c in method_cols if c in agg.columns and not agg[c].isna().all()]
corr_df = pd.DataFrame(index=avail, columns=avail, dtype=float)
for a in avail:
    for b in avail:
        rho, _ = spearmanr(agg[a], agg[b])
        corr_df.loc[a, b] = rho
corr_df = corr_df.astype(float).round(4)
corr_df.index.name = 'method'

# Config sheet
cfg_sheet = pd.DataFrame([
    {'key': 'INTERPRET_TARGET',   'value': str(INTERPRET_TARGET)},
    {'key': 'target_folds',       'value': ', '.join(target_folds)},
    {'key': 'n_folds_aggregated', 'value': len(target_folds)},
    {'key': 'TOP_K',              'value': TOP_K},
    {'key': 'IG_STEPS',           'value': IG_STEPS},
    {'key': 'BATCH_SIZE',         'value': BATCH_SIZE},
    {'key': 'RUN_OCCLUSION',      'value': RUN_OCCLUSION},
    {'key': 'DRIVE_RESULTS_DIR',  'value': str(DRIVE_RESULTS_DIR)},
    {'key': 'output_dir',         'value': str(out_dir)},
    {'key': 'timestamp',          'value': datetime.now().isoformat(timespec='seconds')},
])

with pd.ExcelWriter(xlsx_path, engine='xlsxwriter') as writer:
    top_sheet.to_excel(writer, sheet_name='top_' + str(TOP_K), index=False)
    full_sheet.to_excel(writer, sheet_name='node_importance', index=False)
    summary_df.to_excel(writer, sheet_name='fold_summary', index=False)
    corr_df.to_excel(writer, sheet_name='method_correlations')
    if per_fold is not None:
        per_fold.to_excel(writer, sheet_name='per_fold_long', index=False)
    cfg_sheet.to_excel(writer, sheet_name='config', index=False)

    # Light formatting: freeze header row, autosize columns, bold header
    workbook = writer.book
    bold_header = workbook.add_format({'bold': True, 'bg_color': '#D9E1F2', 'border': 1})
    num_fmt    = workbook.add_format({'num_format': '0.0000'})

    for sheet_name, df in [
        ('top_' + str(TOP_K), top_sheet),
        ('node_importance', full_sheet),
        ('fold_summary', summary_df),
        ('config', cfg_sheet),
    ]:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        # Bold + colored header
        for col_idx, col in enumerate(df.columns):
            ws.write(0, col_idx, str(col), bold_header)
        # Auto-width per column based on content
        for col_idx, col in enumerate(df.columns):
            try:
                max_len = max(
                    len(str(col)),
                    df[col].astype(str).map(len).max() if len(df) else 0,
                )
            except Exception:
                max_len = len(str(col))
            ws.set_column(col_idx, col_idx, min(max_len + 2, 40))
        # Format numeric columns to 4 decimals where it makes sense
        for col_idx, col in enumerate(df.columns):
            if df[col].dtype.kind in 'fc':
                ws.set_column(col_idx, col_idx, None, num_fmt)

    # Special: format the correlation sheet
    ws = writer.sheets['method_correlations']
    ws.freeze_panes(1, 1)
    for col_idx, col in enumerate(['method'] + list(corr_df.columns)):
        ws.write(0, col_idx, str(col), bold_header)
    ws.set_column(0, 0, 18)
    ws.set_column(1, len(corr_df.columns), 14, num_fmt)

sheet_names = [f'top_{TOP_K}', 'node_importance', 'fold_summary', 'method_correlations', 'config']
if per_fold is not None:
    sheet_names.insert(4, 'per_fold_long')
print(f'\n→ Saved Excel workbook to {xlsx_path}')
print('   Sheets: ' + ', '.join(sheet_names))
print(f'   File size: {xlsx_path.stat().st_size / 1024:.1f} KB')

## 11. Edge importance — source → destination strength

Two GNN-native methods that give you a `(roi_src, roi_dst, score)` table:

* **GAT attention** — per-edge α from each GATv2Conv layer (averaged across heads and layers, then across windows). This is the model's *own* measure of how much each connection mattered for the message passing.
* **Edge gradient** — `|∂ŷ / ∂edge_attr|`. How much would the prediction change if I tweaked this edge's input weight slightly. Reflects the full pipeline.

Both are accumulated **undirected** (edge `i↔j` and `j↔i` are merged) and aggregated across all test windows of the chosen fold(s).

Skipping edge-occlusion for now — with ~1k edges per window it's prohibitively slow. GAT attention + gradient already give you the headline picture.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np


def _gat_forward_basic_capture(model, batch, n_rois):
    """FBNetGenFromGraph forward with per-layer GAT attention captured."""
    x = model.node_encoder(batch.x)
    edge_index = batch.edge_index
    ea = batch.edge_attr if hasattr(batch, 'edge_attr') else None
    if model.refine_graph and edge_index.size(1) > 0:
        src, dst = edge_index[0], edge_index[1]
        q = model.W_q(x[src]); k = model.W_k(x[dst])
        attn_score = torch.sigmoid(model.edge_scorer(q * k))
        ea_in = ea if (ea is None or ea.dim() == 2) else ea.unsqueeze(-1)
        edge_weight = ea_in * attn_score if ea_in is not None else attn_score
    else:
        edge_weight = ea if (ea is None or ea.dim() == 2) else ea.unsqueeze(-1)

    p = model.predictor
    h = p.input_proj(x)
    layer_alphas = []
    for i, (gat, norm) in enumerate(zip(p.gat_layers, p.norms)):
        h_res = h
        h, (_, alpha) = gat(h, edge_index, edge_attr=edge_weight,
                            return_attention_weights=True)
        layer_alphas.append(alpha)         # (E, n_heads)
        h = F.elu(h); h = norm(h)
        if i > 0:
            h = h + h_res
    return edge_index, edge_weight, layer_alphas


def _gat_forward_enhanced_capture(model, batch, n_rois):
    x = model.encoder(batch.x)
    edge_index = batch.edge_index
    ea = batch.edge_attr if hasattr(batch, 'edge_attr') else None
    if edge_index.size(1) > 0:
        src, dst = edge_index[0], edge_index[1]
        q = model.W_q(x[src]); k = model.W_k(x[dst])
        attn_score = torch.sigmoid(model.edge_scorer(q * k))
        ea_in = ea if (ea is None or ea.dim() == 2) else ea.unsqueeze(-1)
        edge_weight = ea_in * attn_score if ea_in is not None else attn_score
    else:
        edge_weight = ea

    p = model.gnn_predictor
    h = p.input_proj(x)
    layer_alphas = []
    iterables = zip(p.gat_layers, p.pairnorms, p.layer_norms, p.dropouts, p.residual_projs)
    for i, (gat, pairnorm, ln, drop, res_proj) in enumerate(iterables):
        identity = res_proj(h)
        h, (_, alpha) = gat(h, edge_index, edge_attr=edge_weight,
                            return_attention_weights=True)
        layer_alphas.append(alpha)
        h = pairnorm(h); h = drop(h); h = h + identity; h = ln(h)
        if i < p.n_layers - 1:
            h = F.relu(h)
    return edge_index, edge_weight, layer_alphas


def collect_edge_attention(model, arch, loader, device, n_rois=268):
    """
    Returns flat (N*N,) arrays: sum and count of GAT alpha per undirected (lo, hi) ROI pair.
    Alpha is averaged across layers and heads before accumulation.
    """
    edge_sum = np.zeros(n_rois * n_rois, dtype=np.float64)
    edge_cnt = np.zeros(n_rois * n_rois, dtype=np.int64)
    fwd = _gat_forward_basic_capture if arch == 'basic' else _gat_forward_enhanced_capture

    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            edge_index, _, layer_alphas = fwd(model, batch, n_rois)
            # Stack (n_layers, E, n_heads) -> mean across layers and heads -> (E,)
            stacked = torch.stack([a.mean(dim=1) for a in layer_alphas], dim=0)
            mean_alpha = stacked.mean(dim=0)

            # Map global node idx -> ROI idx (each graph has exactly n_rois nodes, canonical order)
            graph_id = batch.batch[edge_index[0]]
            roi_src = (edge_index[0] - graph_id * n_rois).cpu().numpy()
            roi_dst = (edge_index[1] - graph_id * n_rois).cpu().numpy()
            attn_np = mean_alpha.detach().cpu().numpy().astype(np.float64)

            lo = np.minimum(roi_src, roi_dst)
            hi = np.maximum(roi_src, roi_dst)
            flat = lo * n_rois + hi
            edge_sum += np.bincount(flat, weights=attn_np, minlength=n_rois * n_rois)
            edge_cnt += np.bincount(flat, minlength=n_rois * n_rois)
    return edge_sum, edge_cnt


def collect_edge_gradient(model, loader, device, n_rois=268):
    """|d ŷ / d edge_attr|, summed over feature dim, accumulated per undirected (lo, hi) ROI pair."""
    edge_sum = np.zeros(n_rois * n_rois, dtype=np.float64)
    edge_cnt = np.zeros(n_rois * n_rois, dtype=np.int64)

    model.eval()
    for batch in loader:
        batch = batch.to(device)
        ea_orig = batch.edge_attr.detach().clone()
        ea = ea_orig.clone().requires_grad_(True)
        batch.edge_attr = ea
        out = model(batch)
        grads = torch.autograd.grad(out.sum(), ea, retain_graph=False)[0]
        if grads.dim() > 1:
            grad_per_edge = grads.abs().sum(dim=-1)
        else:
            grad_per_edge = grads.abs()
        batch.edge_attr = ea_orig

        edge_index = batch.edge_index
        graph_id = batch.batch[edge_index[0]]
        roi_src = (edge_index[0] - graph_id * n_rois).cpu().numpy()
        roi_dst = (edge_index[1] - graph_id * n_rois).cpu().numpy()
        grad_np = grad_per_edge.detach().cpu().numpy().astype(np.float64)

        lo = np.minimum(roi_src, roi_dst)
        hi = np.maximum(roi_src, roi_dst)
        flat = lo * n_rois + hi
        edge_sum += np.bincount(flat, weights=grad_np, minlength=n_rois * n_rois)
        edge_cnt += np.bincount(flat, minlength=n_rois * n_rois)
    return edge_sum, edge_cnt


def collect_edge_attr_baseline(loader, n_rois=268):
    """Mean LDW edge weight per (lo, hi) ROI pair — useful for context/sanity."""
    s = np.zeros(n_rois * n_rois, dtype=np.float64)
    c = np.zeros(n_rois * n_rois, dtype=np.int64)
    for batch in loader:
        ei = batch.edge_index
        ea = batch.edge_attr
        if ea.dim() == 2:
            ea = ea.squeeze(-1)
        graph_id = batch.batch[ei[0]]
        roi_src = (ei[0] - graph_id * n_rois).cpu().numpy()
        roi_dst = (ei[1] - graph_id * n_rois).cpu().numpy()
        ea_np = ea.detach().cpu().numpy().astype(np.float64)
        lo = np.minimum(roi_src, roi_dst); hi = np.maximum(roi_src, roi_dst)
        flat = lo * n_rois + hi
        s += np.bincount(flat, weights=ea_np, minlength=n_rois * n_rois)
        c += np.bincount(flat, minlength=n_rois * n_rois)
    return s, c

print('Edge-importance functions defined: collect_edge_attention, collect_edge_gradient, collect_edge_attr_baseline')

In [ ]:
import time
import pandas as pd

def run_edge_for_fold(fold_name, n_rois=268):
    print(f'\n{"="*60}\n{fold_name} - edge importance\n{"="*60}')
    ckpt_path = RESULTS_DIR / fold_name / 'fbnetgen_best.pt'
    fold_path = LOCAL_FOLD_DIR / f'{fold_name}.pkl'

    train_g, val_g, test_g, info = load_graphs_with_normalization(
        str(fold_path), normalize_method='standard'
    )
    in_dim = train_g[0].x.size(-1)
    test_loader = DataLoader(test_g, batch_size=BATCH_SIZE, shuffle=False)

    model, cfg, arch = load_fbnetgen(ckpt_path, in_dim=in_dim, device=device)

    t0 = time.time()
    attn_sum, attn_cnt = collect_edge_attention(model, arch, test_loader, device, n_rois=n_rois)
    print(f'  GAT attention done   ({time.time()-t0:.1f}s)')

    t0 = time.time()
    grad_sum, grad_cnt = collect_edge_gradient(model, test_loader, device, n_rois=n_rois)
    print(f'  edge gradient done   ({time.time()-t0:.1f}s)')

    t0 = time.time()
    ldw_sum,  ldw_cnt  = collect_edge_attr_baseline(test_loader, n_rois=n_rois)
    print(f'  LDW baseline done    ({time.time()-t0:.1f}s)')

    return {
        'fold': fold_name, 'arch': arch,
        'attn_sum': attn_sum, 'attn_cnt': attn_cnt,
        'grad_sum': grad_sum, 'grad_cnt': grad_cnt,
        'ldw_sum':  ldw_sum,  'ldw_cnt':  ldw_cnt,
    }


saved_edge_csv = out_dir / f'edge_importance_{target_suffix}.csv'
n_rois = 268

if USE_EXISTING_INTERPRETABILITY and saved_edge_csv.exists():
    print(f'\nLoading existing edge importance: {saved_edge_csv}')
    edge_df = pd.read_csv(saved_edge_csv)
    per_fold_edges = []
    print(f'Loaded {len(edge_df):,} edge rows; skipping slow edge XAI recomputation.')
else:
    agg_attn_sum = np.zeros(n_rois * n_rois, dtype=np.float64)
    agg_attn_cnt = np.zeros(n_rois * n_rois, dtype=np.int64)
    agg_grad_sum = np.zeros(n_rois * n_rois, dtype=np.float64)
    agg_grad_cnt = np.zeros(n_rois * n_rois, dtype=np.int64)
    agg_ldw_sum  = np.zeros(n_rois * n_rois, dtype=np.float64)
    agg_ldw_cnt  = np.zeros(n_rois * n_rois, dtype=np.int64)

    per_fold_edges = []
    for fname in target_folds:
        r = run_edge_for_fold(fname, n_rois=n_rois)
        agg_attn_sum += r['attn_sum']; agg_attn_cnt += r['attn_cnt']
        agg_grad_sum += r['grad_sum']; agg_grad_cnt += r['grad_cnt']
        agg_ldw_sum  += r['ldw_sum'];  agg_ldw_cnt  += r['ldw_cnt']
        per_fold_edges.append(r)

    present = agg_attn_cnt > 0
    flat_idx = np.where(present)[0]
    src = flat_idx // n_rois
    dst = flat_idx %  n_rois

    mean_attn = np.zeros_like(agg_attn_sum); mean_attn[present] = agg_attn_sum[present] / agg_attn_cnt[present]
    mean_grad = np.zeros_like(agg_grad_sum); mean_grad[present] = agg_grad_sum[present] / np.maximum(agg_grad_cnt[present], 1)
    mean_ldw  = np.zeros_like(agg_ldw_sum);  mean_ldw[present]  = agg_ldw_sum[present]  / np.maximum(agg_ldw_cnt[present], 1)

    edge_df = pd.DataFrame({
        'roi_src':   src,
        'roi_dst':   dst,
        'gat_attn':  mean_attn[present],
        'edge_grad': mean_grad[present],
        'ldw_weight': mean_ldw[present],
        'n_windows': agg_attn_cnt[present],
    })
    edge_df = edge_df[edge_df.roi_src != edge_df.roi_dst].reset_index(drop=True)

    edge_df['attn_rank'] = edge_df['gat_attn'].rank(ascending=False, method='min').astype(int)
    edge_df['grad_rank'] = edge_df['edge_grad'].rank(ascending=False, method='min').astype(int)
    edge_df['mean_rank'] = (edge_df['attn_rank'] + edge_df['grad_rank']) / 2
    edge_df = edge_df.sort_values('mean_rank').reset_index(drop=True)
    edge_df.insert(0, 'consensus_rank', np.arange(1, len(edge_df) + 1))
    if 'roi_label_df' in globals():
        src_labels = roi_label_df.add_prefix('src_').rename(columns={'src_roi_index': 'roi_src'})
        dst_labels = roi_label_df.add_prefix('dst_').rename(columns={'dst_roi_index': 'roi_dst'})
        edge_df = edge_df.merge(src_labels, on='roi_src', how='left').merge(dst_labels, on='roi_dst', how='left')

print(f'\n=== {len(edge_df):,} unique undirected edges across {len(target_folds)} fold(s) ===')
print(f'\nTop {TOP_K} edges by consensus rank:')
edge_show_cols = ['consensus_rank', 'roi_src', 'src_node_no', 'src_roi_label', 'roi_dst', 'dst_node_no', 'dst_roi_label', 'gat_attn', 'edge_grad', 'ldw_weight', 'attn_rank', 'grad_rank', 'n_windows']
edge_show_cols = [c for c in edge_show_cols if c in edge_df.columns]
print(edge_df.head(TOP_K)[edge_show_cols].to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Build a 268x268 mean-attention heat-map (undirected — symmetric)
heat = np.zeros((n_rois, n_rois), dtype=np.float64)
heat[edge_df.roi_src.values, edge_df.roi_dst.values] = edge_df.gat_attn.values
heat = np.maximum(heat, heat.T)

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(heat, cmap='magma', square=True, cbar_kws={'label': 'mean GAT attention'}, ax=ax)
ax.set_title(f'Edge attention (Shen-268)  —  fold(s): {", ".join(target_folds)}')
ax.set_xlabel('ROI dst'); ax.set_ylabel('ROI src')
plt.tight_layout()
plt.savefig(out_dir / 'edge_attention_heatmap.png', dpi=150)
plt.show()

# 2. Top-K edges as a horizontal bar chart
topE = edge_df.head(TOP_K).copy()
if 'src_node_no' in topE.columns and 'dst_node_no' in topE.columns:
    labels = [f'{int(sn):3d} <-> {int(dn):3d}' for sn, dn in zip(topE.src_node_no, topE.dst_node_no)]
else:
    labels = [f'{int(s):3d} <-> {int(d):3d}' for s, d in zip(topE.roi_src, topE.roi_dst)]
fig, ax = plt.subplots(figsize=(11, max(6, TOP_K * 0.3)))
ax.barh(labels[::-1], topE.gat_attn[::-1], color='#cc4444', label='GAT attn')
ax.set_xlabel('mean GAT attention')
ax.set_title(f'Top {TOP_K} edges (consensus) — folds: {", ".join(target_folds)}')
plt.tight_layout()
plt.savefig(out_dir / 'top_edges_bar.png', dpi=150)
plt.show()

# 3. CSV
edge_csv = out_dir / f'edge_importance_{"-".join(target_folds[:1])}{("_+" + str(len(target_folds)-1)) if len(target_folds) > 1 else ""}.csv'
edge_df.to_csv(edge_csv, index=False)
print(f'\n→ Saved {len(edge_df):,} edges to {edge_csv}')

# 4. Append to the existing .xlsx as new sheets (re-open via openpyxl-friendly merge)
from openpyxl import load_workbook
xlsx_path = out_dir / f'fbnetgen_interpretability_{"-".join(target_folds[:1])}{("_+" + str(len(target_folds)-1)) if len(target_folds) > 1 else ""}.xlsx'

if xlsx_path.exists():
    # Use openpyxl to append without rewriting existing sheets
    with pd.ExcelWriter(xlsx_path, engine='openpyxl', mode='a',
                        if_sheet_exists='replace') as writer:
        edge_df.head(TOP_K).to_excel(writer, sheet_name=f'top_{TOP_K}_edges', index=False)
        edge_df.to_excel(writer, sheet_name='edge_importance', index=False)
    print(f'→ Appended sheets [top_{TOP_K}_edges, edge_importance] to {xlsx_path.name}')
else:
    # Fallback: write a separate file
    edge_xlsx = out_dir / f'fbnetgen_edge_importance_{"-".join(target_folds[:1])}{("_+" + str(len(target_folds)-1)) if len(target_folds) > 1 else ""}.xlsx'
    with pd.ExcelWriter(edge_xlsx, engine='xlsxwriter') as writer:
        edge_df.head(TOP_K).to_excel(writer, sheet_name=f'top_{TOP_K}_edges', index=False)
        edge_df.to_excel(writer, sheet_name='edge_importance', index=False)
    print(f'→ Wrote separate xlsx: {edge_xlsx}')

# 5. Hub summary — total incident edge importance per ROI
hub_df = pd.DataFrame({
    'roi_index': np.arange(n_rois),
    'sum_gat_attn':  np.bincount(edge_df.roi_src.values, weights=edge_df.gat_attn.values, minlength=n_rois)
                   + np.bincount(edge_df.roi_dst.values, weights=edge_df.gat_attn.values, minlength=n_rois),
    'sum_edge_grad': np.bincount(edge_df.roi_src.values, weights=edge_df.edge_grad.values, minlength=n_rois)
                   + np.bincount(edge_df.roi_dst.values, weights=edge_df.edge_grad.values, minlength=n_rois),
    'degree':        np.bincount(edge_df.roi_src.values, minlength=n_rois)
                   + np.bincount(edge_df.roi_dst.values, minlength=n_rois),
})
hub_df['attn_rank'] = hub_df['sum_gat_attn'].rank(ascending=False, method='min').astype(int)
hub_df['grad_rank'] = hub_df['sum_edge_grad'].rank(ascending=False, method='min').astype(int)
hub_df['mean_rank'] = (hub_df['attn_rank'] + hub_df['grad_rank']) / 2
hub_df = hub_df.sort_values('mean_rank').reset_index(drop=True)
hub_df.insert(0, 'hub_rank', np.arange(1, len(hub_df) + 1))
if 'roi_label_df' in globals():
    hub_df = hub_df.merge(roi_label_df[label_cols], on='roi_index', how='left')

print(f'\nTop {TOP_K} hub ROIs (highest total incident edge importance):')
hub_show_cols = ['hub_rank', 'roi_index', 'node_no', 'roi_label', 'sum_gat_attn', 'sum_edge_grad', 'degree', 'attn_rank', 'grad_rank']
hub_show_cols = [c for c in hub_show_cols if c in hub_df.columns]
print(hub_df.head(TOP_K)[hub_show_cols].to_string(index=False))

# Also append hub sheet
if xlsx_path.exists():
    with pd.ExcelWriter(xlsx_path, engine='openpyxl', mode='a',
                        if_sheet_exists='replace') as writer:
        hub_df.to_excel(writer, sheet_name='edge_hubs', index=False)
    print(f'→ Appended sheet [edge_hubs] to {xlsx_path.name}')

## 12. Brain visualisation (nilearn)

Plot the importance results onto a brain using MNI coordinates of each Shen-268 ROI.

Produces (under `<output_dir>/brain_plots/`):
* `brain_node_markers_<view>.png` — node importance as colored markers (ortho / sagittal / coronal / axial / 4-panel anatomical).
* `brain_top<K>_nodes_<view>.png` — top-K ROIs highlighted as red spheres.
* `brain_connectome_top<K>_<view>.png` — top-K edges drawn between MNI centroids.
* `brain_grid_summary.png` — compact 6-panel summary (front / side / top / nodes / two connectome views).

Requires a CSV of Shen-268 MNI coordinates. You can:
* Set `ATLAS_CSV_PATH` (in the CONFIG cell) to a CSV with columns `x_mni, y_mni, z_mni` (or `x, y, z`) — one row per ROI in canonical 0..267 order, optionally with a `region_name` / `network` column.
* Or place such a CSV at `GNN-mri/data/shen268_coords.csv` or `GNN-mri/data/shen_268_coords.csv` and the notebook will pick it up automatically.

In [ ]:
# Install nilearn
import subprocess
def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
_run('pip install -q nilearn')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Load Shen-268 MNI centroids (and optional network labels) ─────────
def _load_shen_coords():
    candidates = []
    if 'ATLAS_CSV_PATH' in globals() and ATLAS_CSV_PATH:
        candidates.append(Path(ATLAS_CSV_PATH))
    candidates += [
        Path('/content/drive/MyDrive/GNN-mri/data/shen268_coords.csv'),
        Path('/content/drive/MyDrive/GNN-mri/data/shen_268_coords.csv'),
        Path('/content/drive/MyDrive/GNN-mri/data/shen268_labels.csv'),
        Path('/content/drive/MyDrive/GNN-mri/data/shen_268_labels.csv'),
        Path('/content/GNN-mri/data/shen268_coords.csv'),
        Path('/content/GNN-mri/data/shen_268_coords.csv'),
        Path('/content/GNN-mri/data/shen268_labels.csv'),
        Path('/content/GNN-mri/data/shen_268_labels.csv'),
        Path('/content/GNN-mri/data/shen_268_parcellation_networklabels.csv'),
        Path('/content/folds_data/shen268_coords.csv'),
        Path('/content/folds_data/shen_268_coords.csv'),
    ]
    for p in candidates:
        if not p or not p.exists():
            continue
        df = pd.read_csv(p)
        cols = {c.lower().strip(): c for c in df.columns}
        x_col = next((cols[k] for k in ['x_mni', 'x', 'mni_x'] if k in cols), None)
        y_col = next((cols[k] for k in ['y_mni', 'y', 'mni_y'] if k in cols), None)
        z_col = next((cols[k] for k in ['z_mni', 'z', 'mni_z'] if k in cols), None)
        if not (x_col and y_col and z_col):
            continue
        if len(df) != 268:
            print(f'  {p.name}: has {len(df)} rows, not 268 - skipping')
            continue

        nodeno_col   = next((cols[k] for k in ['nodeno', 'node_no'] if k in cols), None)
        zero_idx_col = None
        if nodeno_col is None:
            zero_idx_col = next((cols[k] for k in ['roi_index', 'node_index', 'roi', 'index'] if k in cols), None)
        if nodeno_col is not None:
            df = df.sort_values(nodeno_col).reset_index(drop=True)
            print(f'  {p.name}: NodeNo (1-indexed) detected, sorted, mapped to 0..267')
        elif zero_idx_col is not None:
            df = df.sort_values(zero_idx_col).reset_index(drop=True)
            print(f'  {p.name}: 0-indexed node-index detected, sorted')
        else:
            print(f'  {p.name}: no node-index column - assuming canonical 0..267 order')

        coords = df[[x_col, y_col, z_col]].to_numpy(dtype=float)
        net_col = next((cols[k] for k in ['network', 'networks', 'net_name'] if k in cols), None)
        reg_col = next((cols[k] for k in ['region_name', 'region', 'name', 'label'] if k in cols), None)
        networks_csv = df[net_col].astype(str).tolist() if net_col else None
        regions_csv  = df[reg_col].astype(str).tolist() if reg_col else None
        print(f'  Atlas loaded: {p}  ({len(df)} rows)')
        return coords, networks_csv, regions_csv
    return None, None, None


centroids, networks_csv, regions_csv = _load_shen_coords()
if centroids is None:
    raise FileNotFoundError(
        "No Shen-268 coordinate CSV found. "
        "Provide one with columns (NodeNo, x_mni, y_mni, z_mni) [+ optional region_name, network] "
        "either in the ATLAS_CSV_PATH config var, /content/drive/MyDrive/GNN-mri/data/shen268_coords.csv, "
        "or /content/GNN-mri/data/shen268_coords.csv"
    )
print(f'  Centroid bounds:  x [{centroids[:,0].min():.1f}, {centroids[:,0].max():.1f}], '
      f'y [{centroids[:,1].min():.1f}, {centroids[:,1].max():.1f}], '
      f'z [{centroids[:,2].min():.1f}, {centroids[:,2].max():.1f}]')


# ── Always: derive coarse hemisphere + lobe (anatomical detail label) ────
def _derive_hemi_lobe(coords):
    hemis, lobes = [], []
    for x, y, z in coords:
        h = 'Left' if x < -2 else ('Right' if x > 2 else 'Midline')
        if z < -25:
            lobe = 'Cerebellum/BrainStem'
        elif y > 30 and z > -10:
            lobe = 'Frontal'
        elif y > 0 and z > 35:
            lobe = 'Frontal'
        elif y < -50 and abs(x) > 25:
            lobe = 'Occipital' if z > -5 else 'Cerebellum/Occipital'
        elif y < -25 and z > 20:
            lobe = 'Parietal'
        elif y < -25 and z < 5 and abs(x) > 20:
            lobe = 'Temporal'
        elif abs(x) < 25 and -20 < y < 20 and z < 10:
            lobe = 'Subcortical/Insular'
        else:
            lobe = 'Other/Limbic'
        hemis.append(h); lobes.append(lobe)
    return hemis, lobes

hemispheres, lobes = _derive_hemi_lobe(centroids)

# region_detail: always available, derived from MNI coords ('Left Frontal', etc.)
# region_overall: starts as same; cell 29 overwrites with network if it can fetch them.
region_detail  = [f'{h} {l}' for h, l in zip(hemispheres, lobes)]
region_overall = list(region_detail)        # fallback if no network labels

# Backwards compat with older code that read 'regions' / 'networks'
networks = list(networks_csv) if networks_csv else None
regions  = list(regions_csv)  if regions_csv  else None

from collections import Counter
print()
print('Auto-derived coarse anatomy (region_detail):')
for lb, n in sorted(Counter(lobes).items(), key=lambda kv: -kv[1]):
    print(f'  {lb:30s}  {n:3d}')


### Optional: fetch Shen-268 network labels from a verified public source

There is **no single canonical** region-name CSV for Shen-268 — different groups publish different annotations. The most widely cited one is the **8-network labelling** from Finn et al. 2015 (Nature Neuroscience), made available by the [canlab](https://github.com/canlab/Neuroimaging_Pattern_Masks) lab on GitHub.

The cell below downloads `shen_268_parcellation_networklabels.csv` from the canlab repository and merges it into the existing `regions` and `networks` lists. After this cell runs, your tooltips will say e.g. `ROI 7 | Left Frontal | Default Mode | imp=0.69` instead of just `ROI 7 | imp=0.69`.

**Network names** (standard Shen/Finn 8-network convention; counts are printed from the CSV at runtime):

| net | name |
|---|---|
| 1 | Medial Frontal |
| 2 | Frontoparietal |
| 3 | Default Mode |
| 4 | Subcortical/Cerebellum |
| 5 | Motor |
| 6 | Visual I |
| 7 | Visual II |
| 8 | Visual Association |

If the GitHub fetch fails (rare — runtime offline, etc.), the cell prints the error and continues with the auto-derived `Right/Left + Lobe` labels from cell 27. No crash.

**URL used (verified working):**
```
https://raw.githubusercontent.com/canlab/Neuroimaging_Pattern_Masks/
   master/Atlases_and_parcellations/
   2013_Shen_Constable_NIMG_268_parcellation/shen_268_parcellation_networklabels.csv
```


In [ ]:
# Optional: fetch Shen-268 8-network labels and assign to region_overall.
# Skip this cell to keep region_overall = region_detail (Right/Left + Lobe).
import urllib.request, ssl, io, pandas as pd

PRIMARY_URL = (
    "https://raw.githubusercontent.com/canlab/Neuroimaging_Pattern_Masks/master/"
    "Atlases_and_parcellations/2013_Shen_Constable_NIMG_268_parcellation/"
    "shen_268_parcellation_networklabels.csv"
)
NET_NAMES = {
    1: "Medial Frontal",
    2: "Frontoparietal",
    3: "Default Mode",
    4: "Subcortical/Cerebellum",
    5: "Motor",
    6: "Visual I",
    7: "Visual II",
    8: "Visual Association",
}

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

def _read_shen_network_csv(raw_text):
    try:
        tmp = pd.read_csv(io.StringIO(raw_text))
        cols = {str(c).lower().strip() for c in tmp.columns}
        if len(tmp) == 268 and {'node', 'network'}.issubset(cols):
            return tmp
    except Exception:
        pass
    rows = []
    for token in raw_text.replace('Node,Network', '').replace('\n', ' ').split():
        if ',' not in token:
            continue
        a, b = token.strip().strip(',').split(',', 1)
        try:
            rows.append((int(a), int(b)))
        except ValueError:
            continue
    if len(rows) != 268:
        raise ValueError(f'expected 268 node/network rows, got {len(rows)}')
    return pd.DataFrame(rows, columns=['Node', 'Network'])

try:
    with urllib.request.urlopen(PRIMARY_URL, context=ctx, timeout=15) as r:
        raw = r.read().decode("utf-8-sig", errors="replace")
    df = _read_shen_network_csv(raw)
    assert len(df) == 268, f"expected 268 rows, got {len(df)}"
    df = df.sort_values("Node").reset_index(drop=True)
    raw_nets = df["Network"].astype(int).tolist()
    networks = [NET_NAMES.get(n, f"Net{n}") for n in raw_nets]

    # region_overall = network name (most informative single overall label)
    # region_detail  = hemisphere + coarse lobe (already set in cell 27)
    region_overall = list(networks)

    from collections import Counter
    print(f"  Fetched: {PRIMARY_URL.split('/')[-1]} (268 rows)")
    print("  region_overall = network name (canlab fetch)")
    print("  region_detail  = hemisphere + coarse lobe (auto-derived)")
    print()
    print("  Network counts (region_overall):")
    for net, n in sorted(Counter(networks).items(), key=lambda kv: -kv[1]):
        print(f"    {net:25s}  {n:3d}")

    # Backwards compat: keep 'regions' as the joined string for any older code
    regions = [f"{d} | {o}" for d, o in zip(region_detail, region_overall)]
except Exception as e:
    print(f"  Could not fetch network labels ({type(e).__name__}: {e})")
    print("  Keeping region_overall = region_detail (Right/Left + Lobe).")
    networks = None
    regions  = list(region_detail)


In [ ]:
from nilearn import plotting

# Auto-recover `agg` from saved CSV if not in memory (e.g. after reconnect)
if 'agg' not in globals():
    print('`agg` not in memory - loading from saved node_importance CSV')
    import pandas as pd
    csvs = sorted(out_dir.glob('node_importance_*.csv'))
    if not csvs:
        raise FileNotFoundError(
            f"No node_importance_*.csv in {out_dir}. "
            "Run section 8 (node importance) at least once before plotting."
        )
    agg = pd.read_csv(csvs[-1])
    if 'consensus_z' not in agg.columns:
        method_cols = ['pool_gate', 'saliency', 'integrated_grad', 'occlusion']
        z_cols = []
        for c in method_cols:
            if c in agg.columns and not agg[c].isna().all():
                zc = c + '_z'
                mu, sd = agg[c].mean(), agg[c].std() + 1e-12
                agg[zc] = (agg[c] - mu) / sd
                z_cols.append(zc)
        agg['consensus_z'] = agg[z_cols].mean(axis=1)
    print(f'  loaded {len(agg)} rows from {csvs[-1].name}')

# Pick which node-importance signal to plot
NODE_PLOT_METRIC = 'consensus_z'

if NODE_PLOT_METRIC in agg.columns:
    node_imp = agg.set_index('roi_index')[NODE_PLOT_METRIC].reindex(range(268)).fillna(0).to_numpy()
else:
    node_imp = agg.set_index('roi_index')['consensus_z'].reindex(range(268)).fillna(0).to_numpy()

print(f'Plotting node importance metric: {NODE_PLOT_METRIC}')
print(f'  range: [{node_imp.min():.4f}, {node_imp.max():.4f}]')

top_k = TOP_K
top_idx    = np.argsort(node_imp)[::-1][:top_k]
top_scores = node_imp[top_idx]

print(f'\nTop {top_k} ROIs by {NODE_PLOT_METRIC}:')
hdr = f'  {"rank":>4} {"ROI":>4}  {"x_mni":>6}  {"y_mni":>6}  {"z_mni":>6}  {"score":>8}'
if 'region_overall' in globals() and region_overall: hdr += f'  {"region_overall":<22}'
if 'region_detail'  in globals() and region_detail:  hdr += f'  {"region_detail":<22}'
print(hdr)
print('  ' + '-' * (len(hdr) - 2))
for r, idx in enumerate(top_idx):
    line = f'  {r+1:>4} {idx:>4}  {centroids[idx,0]:+6.1f}  {centroids[idx,1]:+6.1f}  {centroids[idx,2]:+6.1f}  {node_imp[idx]:>8.4f}'
    if 'region_overall' in globals() and region_overall: line += f'  {region_overall[idx]:<22}'
    if 'region_detail'  in globals() and region_detail:  line += f'  {region_detail[idx]:<22}'
    print(line)

# Build node-display arrays
node_size  = np.full(268, 12.0)
top_norm   = (top_scores - top_scores.min()) / (top_scores.max() - top_scores.min() + 1e-12)
node_size[top_idx] = 60 + 200 * top_norm

node_color_marker = node_imp.copy()
node_color_top    = ['#cccccc'] * 268
for r, idx in enumerate(top_idx):
    node_color_top[idx] = '#e63946'

# 1. plot_markers - value-coloured, multiple views
brain_dir = out_dir / 'brain_plots'
brain_dir.mkdir(exist_ok=True)

fig_paths = {}
view_modes = [
    ('ortho', 'Front + side + top (orthogonal)'),
    ('x',     'Sagittal slices'),
    ('y',     'Coronal slices (front)'),
    ('z',     'Axial slices (top)'),
    ('lyrz',  '4-panel: left, front, right, top'),
]
for mode, title in view_modes:
    fig = plotting.plot_markers(
        node_values=node_imp,
        node_coords=centroids,
        display_mode=mode,
        node_cmap='hot',
        node_size=15,
        title=f'Node importance ({NODE_PLOT_METRIC}) - {title}',
        colorbar=True,
    )
    fp = brain_dir / f'brain_node_markers_{mode}.png'
    fig.savefig(fp, dpi=150, bbox_inches='tight')
    fig_paths[f'markers_{mode}'] = fp
    plt.show()
    plt.close('all')

# 2. plot_connectome - top-K nodes highlighted
for mode, title in [('ortho', 'Front + side + top'), ('lyrz', '4-panel anatomical')]:
    fig = plotting.plot_connectome(
        adjacency_matrix=np.zeros((268, 268)),
        node_coords=centroids,
        node_color=node_color_top,
        node_size=node_size,
        display_mode=mode,
        title=f'Top-{top_k} ROIs - {title}',
        colorbar=False,
    )
    fp = brain_dir / f'brain_top{top_k}_nodes_{mode}.png'
    fig.savefig(fp, dpi=150, bbox_inches='tight')
    fig_paths[f'topnodes_{mode}'] = fp
    plt.show()
    plt.close('all')

print(f'\nSaved {len(fig_paths)} brain images to {brain_dir}')


In [ ]:
# Auto-recover `edge_df` from saved CSV if not in memory
if 'edge_df' not in globals():
    print('`edge_df` not in memory - loading from saved edge_importance CSV')
    import pandas as pd
    csvs = sorted(out_dir.glob('edge_importance_*.csv'))
    if not csvs:
        raise FileNotFoundError(
            f"No edge_importance_*.csv in {out_dir}. "
            "Run section 11 (edge importance) at least once before plotting."
        )
    edge_df = pd.read_csv(csvs[-1])
    print(f'  loaded {len(edge_df):,} edges from {csvs[-1].name}')

# 3. Connectome - top-K edges drawn between MNI centroids
TOP_EDGES = 50
EDGE_PLOT_METRIC = 'edge_grad'

if EDGE_PLOT_METRIC == 'consensus_rank':
    top_e = edge_df.nsmallest(TOP_EDGES, 'consensus_rank')
else:
    top_e = edge_df.nlargest(TOP_EDGES, EDGE_PLOT_METRIC)

adj = np.zeros((268, 268), dtype=float)
for _, row in top_e.iterrows():
    s, d = int(row.roi_src), int(row.roi_dst)
    w = float(row[EDGE_PLOT_METRIC]) if EDGE_PLOT_METRIC in row else 1.0
    adj[s, d] = w; adj[d, s] = w

print(f'Drawing top-{TOP_EDGES} edges  (metric={EDGE_PLOT_METRIC})')
print(f'  edge weight range: [{adj[adj>0].min():.6g}, {adj.max():.6g}]')
print(f'  unique nodes touched: {len(set(top_e.roi_src.tolist() + top_e.roi_dst.tolist()))}')

touched = sorted(set(top_e.roi_src.tolist() + top_e.roi_dst.tolist()))
node_size_e = np.full(268, 0.0)
node_size_e[touched] = 30 + 200 * (node_imp[touched] - node_imp.min()) / (node_imp.max() - node_imp.min() + 1e-12)

for mode, title in [('ortho', 'Front + side + top'), ('lyrz', '4-panel anatomical'), ('lzry', '4-panel anatomical (alt)')]:
    fig = plotting.plot_connectome(
        adjacency_matrix=adj,
        node_coords=centroids,
        node_size=node_size_e,
        edge_threshold=None,
        edge_cmap='Reds',
        edge_kwargs={'linewidth': 1.5},
        display_mode=mode,
        colorbar=True,
        title=f'Top-{TOP_EDGES} edges  ({EDGE_PLOT_METRIC}) - {title}',
    )
    fp = brain_dir / f'brain_connectome_top{TOP_EDGES}_{mode}.png'
    fig.savefig(fp, dpi=150, bbox_inches='tight')
    fig_paths[f'connectome_{mode}'] = fp
    plt.show()
    plt.close('all')

print(f'Saved connectome plots to {brain_dir}')


In [ ]:
# ── 4. Compact 6-panel summary figure (presentation-ready) ────────────────
import matplotlib.image as mpimg
from PIL import Image

panels = [
    ('markers_y',      'Front (coronal)'),
    ('markers_x',      'Side (sagittal)'),
    ('markers_z',      'Top (axial)'),
    ('topnodes_lyrz',  f'Top-{top_k} nodes (anatomical)'),
    ('connectome_ortho',  f'Top-{TOP_EDGES} edges (ortho)'),
    ('connectome_lyrz',   f'Top-{TOP_EDGES} edges (anatomical)'),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
for ax, (key, label) in zip(axes.flat, panels):
    if key in fig_paths and fig_paths[key].exists():
        img = mpimg.imread(fig_paths[key])
        ax.imshow(img)
        ax.set_title(label, fontsize=11)
    else:
        ax.text(0.5, 0.5, f'(missing: {key})', ha='center', va='center')
    ax.axis('off')

fig.suptitle(
    f'Brain interpretability  —  fold(s): {", ".join(target_folds)}  '
    f'(test_subj_r mean = {summary_df[summary_df.fold.isin(target_folds)]["test_subj_r"].mean():.3f})',
    fontsize=13,
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
summary_path = brain_dir / 'brain_grid_summary.png'
fig.savefig(summary_path, dpi=160, bbox_inches='tight')
plt.show()
plt.close('all')

print(f'\n→ Brain summary panel saved: {summary_path}')
print(f'\nAll brain plots:')
for k, p in fig_paths.items():
    print(f'  {k:25s}  {p.name}')
print(f'  {"summary":25s}  {summary_path.name}')

## 14. Interactive 3D brain (HTML)

Uses `nilearn.view_markers` + `view_connectome` to produce **interactive 3D brain viewers** you can rotate/zoom/hover. Each is saved as a standalone HTML file and also rendered inline in the Colab cell output.

* `brain_3d_markers.html` - all 268 ROIs as coloured spheres, hover for ROI label/network/importance.
* `brain_3d_connectome_top<N>.html` - top-K edges drawn in 3D between MNI centroids.


In [ ]:
# Interactive 3D brain (nilearn HTML viewers)
# Renders interactive iframes INSIDE this cell's output (drag to rotate, hover for labels).
# Also saves standalone .html files you can share / embed elsewhere.
from nilearn import plotting as nlp
from IPython.display import display
import matplotlib.cm as _cm
import matplotlib.colors as _mc

# ---- Convert numeric importance -> hex colour list (one per ROI) -----------
# view_markers wants colour strings, not a value array (unlike plot_markers).
_norm = _mc.Normalize(vmin=float(node_imp.min()), vmax=float(node_imp.max()))
_cmap = _cm.get_cmap('hot')
marker_colors = [_mc.to_hex(_cmap(_norm(float(v)))) for v in node_imp]

# Size scaled to importance (clip a floor so low-importance nodes are still visible)
_imp_norm = (node_imp - node_imp.min()) / (node_imp.max() - node_imp.min() + 1e-12)
marker_sizes = (5.0 + 18.0 * _imp_norm).tolist()

# ---- 1. Interactive 3D markers (every ROI coloured by importance) -----------
_ro = region_overall if 'region_overall' in globals() and region_overall else None
_rd = region_detail  if 'region_detail'  in globals() and region_detail  else None
labels = [
    f'ROI {i}'
    + (f' | {_ro[i]}' if _ro else '')
    + (f' | {_rd[i]}' if _rd else '')
    + f' | imp={node_imp[i]:.4f}'
    for i in range(268)
]
view_markers = nlp.view_markers(
    marker_coords=centroids,
    marker_color=marker_colors,
    marker_size=marker_sizes,
    marker_labels=labels,
)
markers_html = brain_dir / 'brain_3d_markers.html'
view_markers.save_as_html(str(markers_html))
print(f'Interactive 3D markers   : {markers_html.name}')
display(view_markers)            # renders inline iframe in this Colab cell

# ---- 2. Interactive 3D connectome (top-K edges drawn in 3D) -----------------
# Reuses `adj` and `node_size_e` built in cell 29.
view_connect = nlp.view_connectome(
    adjacency_matrix=adj,
    node_coords=centroids,
    edge_threshold=None,             # already pre-thresholded by cell 29
    edge_cmap='Reds',
    node_size=4.0 + 0.04 * node_size_e,
    linewidth=4.0,
    title=f'Top-{TOP_EDGES} edges  (metric={EDGE_PLOT_METRIC})',
)
connectome_html = brain_dir / f'brain_3d_connectome_top{TOP_EDGES}.html'
view_connect.save_as_html(str(connectome_html))
print(f'Interactive 3D connectome: {connectome_html.name}')
display(view_connect)            # renders inline iframe in this Colab cell

print()
print('Both files saved to:')
print(f'  {markers_html}')
print(f'  {connectome_html}')
print()
print('Inside this notebook: drag the 3D brain to rotate, scroll to zoom, hover for tooltips.')
print('Outside:  open the .html files in any browser, or embed via')
print('          <iframe src="brain_3d_connectome_topN.html" width="100%" height="600"></iframe>')


In [ ]:
# Easy-to-read labeled outputs: CSV + named bar plots
# Run this after node_importance / edge_importance CSVs exist.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Make this cell runnable even if you skipped earlier compute cells.
if 'out_dir' not in globals():
    out_dir = Path(DRIVE_OUTPUT_DIR) if 'DRIVE_OUTPUT_DIR' in globals() else Path('/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability')

if 'target_suffix' not in globals():
    if 'target_folds' in globals():
        def _target_suffix(folds):
            return f'{"-".join(folds[:1])}{("_+" + str(len(folds)-1)) if len(folds) > 1 else ""}'
        target_suffix = _target_suffix(target_folds)
    else:
        # Infer from existing saved CSVs in the interpretability folder.
        node_candidates = sorted(out_dir.glob('node_importance_*.csv'))
        if not node_candidates:
            raise FileNotFoundError(f'No node_importance_*.csv found in {out_dir}. Run interpretability once or check DRIVE_OUTPUT_DIR.')
        node_csv_inferred = node_candidates[-1]
        target_suffix = node_csv_inferred.stem.replace('node_importance_', '', 1)
        target_folds = [target_suffix.split('_+')[0]]
        print(f'Inferred target_suffix from saved CSV: {target_suffix}')

readable_dir = out_dir / 'readable_outputs'
readable_dir.mkdir(parents=True, exist_ok=True)

# Load saved node/edge importance if not already in memory.
node_csv = out_dir / f'node_importance_{target_suffix}.csv'
edge_csv = out_dir / f'edge_importance_{target_suffix}.csv'

if 'agg' not in globals():
    if not node_csv.exists():
        raise FileNotFoundError(f'Missing node importance CSV: {node_csv}')
    agg = pd.read_csv(node_csv)

if 'edge_df' not in globals():
    if edge_csv.exists():
        edge_df = pd.read_csv(edge_csv)
    else:
        edge_df = None
        print(f'No edge importance CSV found at {edge_csv}; node outputs only.')

# Build / reuse Shen labels. This gives ROI number, network, hemisphere/lobe if available.
if 'roi_label_df' not in globals():
    roi_label_df = build_shen268_label_table(n_rois=268)
    label_cols = [c for c in [
        'roi_index', 'node_no', 'roi_name', 'roi_label', 'region_name',
        'region_detail', 'network', 'hemisphere', 'lobe', 'x_mni', 'y_mni', 'z_mni'
    ] if c in roi_label_df.columns]

# Create labeled node table.
node_labeled = agg.copy()
for c in [c for c in label_cols if c != 'roi_index']:
    if c in node_labeled.columns:
        node_labeled = node_labeled.drop(columns=[c])
node_labeled = node_labeled.merge(roi_label_df[label_cols], on='roi_index', how='left')

# Make a short label for plots.
def _short_node_label(row):
    node_no = int(row['node_no']) if pd.notna(row.get('node_no', np.nan)) else int(row['roi_index']) + 1
    parts = [f'ROI {node_no}']
    for c in ['network', 'hemisphere', 'lobe', 'region_detail', 'region_name']:
        val = row.get(c, '')
        if pd.notna(val) and str(val).strip() and str(val).lower() != 'nan':
            parts.append(str(val))
            break
    return ' | '.join(parts)

node_labeled['display_label'] = node_labeled.apply(_short_node_label, axis=1)

# Put readable columns first.
node_first_cols = [
    'consensus_rank', 'roi_index', 'node_no', 'display_label', 'roi_label',
    'network', 'hemisphere', 'lobe', 'region_detail', 'region_name'
]
node_first_cols = [c for c in node_first_cols if c in node_labeled.columns]
node_labeled = node_labeled[node_first_cols + [c for c in node_labeled.columns if c not in node_first_cols]]
node_labeled = node_labeled.sort_values('consensus_rank')

node_out = readable_dir / f'node_importance_labeled_{target_suffix}.csv'
node_labeled.to_csv(node_out, index=False)
print(f'Saved labeled node CSV: {node_out}')

# Named node bar plot.
plot_metric = 'consensus_z' if 'consensus_z' in node_labeled.columns else 'mean_rank'
top_nodes = node_labeled.head(TOP_K).copy()
fig, ax = plt.subplots(figsize=(12, max(7, TOP_K * 0.34)))
if plot_metric == 'mean_rank':
    values = -top_nodes[plot_metric].to_numpy()  # lower rank is better; invert for visual bar length
    xlabel = 'Importance ranking score (higher bar = better rank)'
else:
    values = top_nodes[plot_metric].to_numpy()
    xlabel = plot_metric
ax.barh(top_nodes['display_label'][::-1], values[::-1], color='#4c78a8')
ax.set_xlabel(xlabel)
ax.set_title(f'Top {TOP_K} labeled ROIs - {target_suffix}')
plt.tight_layout()
node_png = readable_dir / f'top_{TOP_K}_rois_labeled_{target_suffix}.png'
plt.savefig(node_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved labeled node plot: {node_png}')

# Create labeled edge table + plot if edge data exists.
if edge_df is not None:
    edge_labeled = edge_df.copy()
    src_labels = roi_label_df[label_cols].add_prefix('src_').rename(columns={'src_roi_index': 'roi_src'})
    dst_labels = roi_label_df[label_cols].add_prefix('dst_').rename(columns={'dst_roi_index': 'roi_dst'})
    edge_labeled = edge_labeled.merge(src_labels, on='roi_src', how='left')
    edge_labeled = edge_labeled.merge(dst_labels, on='roi_dst', how='left')

    def _edge_label(row):
        s_no = int(row['src_node_no']) if pd.notna(row.get('src_node_no', np.nan)) else int(row['roi_src']) + 1
        d_no = int(row['dst_node_no']) if pd.notna(row.get('dst_node_no', np.nan)) else int(row['roi_dst']) + 1
        s_net = row.get('src_network', '')
        d_net = row.get('dst_network', '')
        s = f'ROI {s_no}' + (f' ({s_net})' if pd.notna(s_net) and str(s_net).strip() else '')
        d = f'ROI {d_no}' + (f' ({d_net})' if pd.notna(d_net) and str(d_net).strip() else '')
        return f'{s} <-> {d}'

    edge_labeled['display_label'] = edge_labeled.apply(_edge_label, axis=1)
    edge_first_cols = [
        'consensus_rank', 'roi_src', 'src_node_no', 'roi_dst', 'dst_node_no',
        'display_label', 'src_roi_label', 'dst_roi_label', 'src_network', 'dst_network',
        'gat_attn', 'edge_grad', 'ldw_weight', 'n_windows'
    ]
    edge_first_cols = [c for c in edge_first_cols if c in edge_labeled.columns]
    edge_labeled = edge_labeled[edge_first_cols + [c for c in edge_labeled.columns if c not in edge_first_cols]]
    edge_labeled = edge_labeled.sort_values('consensus_rank')

    edge_out = readable_dir / f'edge_importance_labeled_{target_suffix}.csv'
    edge_labeled.to_csv(edge_out, index=False)
    print(f'Saved labeled edge CSV: {edge_out}')

    top_edges = edge_labeled.head(TOP_K).copy()
    edge_metric = 'edge_grad' if 'edge_grad' in top_edges.columns else 'gat_attn'
    fig, ax = plt.subplots(figsize=(13, max(7, TOP_K * 0.36)))
    ax.barh(top_edges['display_label'][::-1], top_edges[edge_metric][::-1], color='#c44e52')
    ax.set_xlabel(edge_metric)
    ax.set_title(f'Top {TOP_K} labeled edges - {target_suffix}')
    plt.tight_layout()
    edge_png = readable_dir / f'top_{TOP_K}_edges_labeled_{target_suffix}.png'
    plt.savefig(edge_png, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved labeled edge plot: {edge_png}')

    xlsx_out = readable_dir / f'fbnetgen_readable_labeled_{target_suffix}.xlsx'
    with pd.ExcelWriter(xlsx_out, engine='openpyxl') as writer:
        node_labeled.head(TOP_K).to_excel(writer, sheet_name=f'top_{TOP_K}_nodes', index=False)
        node_labeled.to_excel(writer, sheet_name='all_nodes_labeled', index=False)
        edge_labeled.head(TOP_K).to_excel(writer, sheet_name=f'top_{TOP_K}_edges', index=False)
        edge_labeled.to_excel(writer, sheet_name='all_edges_labeled', index=False)
    print(f'Saved readable Excel workbook: {xlsx_out}')
else:
    xlsx_out = readable_dir / f'fbnetgen_readable_labeled_nodes_{target_suffix}.xlsx'
    with pd.ExcelWriter(xlsx_out, engine='openpyxl') as writer:
        node_labeled.head(TOP_K).to_excel(writer, sheet_name=f'top_{TOP_K}_nodes', index=False)
        node_labeled.to_excel(writer, sheet_name='all_nodes_labeled', index=False)
    print(f'Saved readable node-only Excel workbook: {xlsx_out}')


In [ ]:
# Interactive 3D brain with node importance and within/between-category edges
# Change these values, then rerun this cell.
TOP_K_3D_NODES = 50
TOP_K_3D_EDGES = 80
NODE_IMPORTANCE_METRIC = 'consensus_z'
EDGE_IMPORTANCE_METRIC = 'edge_grad'
NODE_CATEGORY_COL = 'network'  # Shen network/category column
EDGE_SCOPE = 'among_top_nodes'  # 'among_top_nodes' or 'top_edges_overall'
NODE_INTENSITY_COLORSCALE = [[0.0, '#eeeeee'], [0.35, '#fff2a8'], [0.7, '#ffd84d'], [1.0, '#f2a900']]  # brighter yellow/gold = more important
SHOW_BRAIN_SURFACE = True
BRAIN_SURFACE_OPACITY = 0.16
SHOW_ALL_ROIS_FAINT = False

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

if 'out_dir' not in globals():
    out_dir = Path(DRIVE_OUTPUT_DIR) if 'DRIVE_OUTPUT_DIR' in globals() else Path('/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability')

if 'target_suffix' not in globals():
    node_candidates = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)
    if not node_candidates:
        raise FileNotFoundError(f'No node_importance_*.csv found in {out_dir}')
    target_suffix = node_candidates[-1].stem.replace('node_importance_', '', 1)
    target_folds = [target_suffix.split('_+')[0]]
    print(f'Inferred target_suffix from saved CSV: {target_suffix}')

if 'agg' not in globals():
    node_csv = out_dir / f'node_importance_{target_suffix}.csv'
    if not node_csv.exists():
        node_csv = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)[-1]
    print(f'Loading node importance: {node_csv}')
    agg = pd.read_csv(node_csv)

if 'edge_df' not in globals():
    edge_csv = out_dir / f'edge_importance_{target_suffix}.csv'
    if not edge_csv.exists():
        edge_candidates = sorted(out_dir.glob('edge_importance_*.csv'), key=lambda p: p.stat().st_mtime)
        edge_csv = edge_candidates[-1] if edge_candidates else None
    if edge_csv and edge_csv.exists():
        print(f'Loading edge importance: {edge_csv}')
        edge_df = pd.read_csv(edge_csv)
    else:
        edge_df = None
        print('No edge_importance_*.csv found; rendering node-only 3D brain.')

if 'centroids' not in globals():
    if '_load_shen_coords' in globals():
        centroids, networks_csv, regions_csv = _load_shen_coords()
    else:
        coord_candidates = [
            Path(globals().get('ATLAS_CSV_PATH', '')),
            Path('/content/drive/MyDrive/GNN-mri/data/shen268_coords.csv'),
            Path('/content/drive/MyDrive/GNN-mri/data/shen_268_coords.csv'),
            Path('/content/GNN-mri/data/shen268_coords.csv'),
            Path('/content/GNN-mri/data/shen_268_coords.csv'),
        ]
        centroids = None
        for p in coord_candidates:
            if p and p.exists():
                dfc = pd.read_csv(p)
                cols = {str(c).lower().strip(): c for c in dfc.columns}
                x_col = next((cols[k] for k in ['x_mni', 'x', 'mni_x'] if k in cols), None)
                y_col = next((cols[k] for k in ['y_mni', 'y', 'mni_y'] if k in cols), None)
                z_col = next((cols[k] for k in ['z_mni', 'z', 'mni_z'] if k in cols), None)
                if x_col and y_col and z_col:
                    nodeno_col = next((cols[k] for k in ['nodeno', 'node_no', 'node'] if k in cols), None)
                    if nodeno_col:
                        dfc = dfc.sort_values(nodeno_col)
                    centroids = dfc[[x_col, y_col, z_col]].to_numpy(dtype=float)
                    print(f'Loaded MNI coordinates from {p}')
                    break
    if centroids is None:
        raise FileNotFoundError('Could not load Shen-268 MNI coordinates for 3D plotting.')

def _clean_category(v):
    if pd.isna(v) or not str(v).strip() or str(v).lower() == 'nan':
        return 'Unknown'
    return str(v).strip()

def _norm(values, low, high):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or abs(mx - mn) < 1e-12:
        return np.full_like(arr, (low + high) / 2, dtype=float)
    return low + (arr - mn) * (high - low) / (mx - mn)

node_df = agg.copy()
node_df['roi_index'] = node_df['roi_index'].astype(int)
if NODE_IMPORTANCE_METRIC not in node_df.columns:
    NODE_IMPORTANCE_METRIC = 'consensus_rank' if 'consensus_rank' in node_df.columns else 'mean_rank'
rank_like = NODE_IMPORTANCE_METRIC.endswith('_rank') or NODE_IMPORTANCE_METRIC in ['consensus_rank', 'mean_rank']
node_df['_node_score_raw'] = pd.to_numeric(node_df[NODE_IMPORTANCE_METRIC], errors='coerce')
node_df['_node_score'] = -node_df['_node_score_raw'] if rank_like else node_df['_node_score_raw']
node_df['_node_magnitude'] = node_df['_node_score'].abs()
node_df['_node_size'] = _norm(node_df['_node_magnitude'].fillna(0), 7, 24)
node_df['_category'] = node_df[NODE_CATEGORY_COL].map(_clean_category) if NODE_CATEGORY_COL in node_df.columns else 'Unknown'
top_nodes = node_df.sort_values('_node_score', ascending=False).head(TOP_K_3D_NODES).copy()
shown_rois = set(top_nodes['roi_index'].astype(int))
node_lookup = node_df.set_index('roi_index').to_dict('index')
cmin, cmax = float(top_nodes['_node_score'].min()), float(top_nodes['_node_score'].max())
categories = sorted(top_nodes['_category'].dropna().unique().tolist())

edge_plot = pd.DataFrame()
if edge_df is not None and len(edge_df):
    edge_plot = edge_df.copy()
    edge_plot['roi_src'] = edge_plot['roi_src'].astype(int)
    edge_plot['roi_dst'] = edge_plot['roi_dst'].astype(int)
    if EDGE_IMPORTANCE_METRIC not in edge_plot.columns:
        EDGE_IMPORTANCE_METRIC = 'consensus_rank' if 'consensus_rank' in edge_plot.columns else ('edge_grad' if 'edge_grad' in edge_plot.columns else 'gat_attn')
    edge_rank_like = EDGE_IMPORTANCE_METRIC.endswith('_rank') or EDGE_IMPORTANCE_METRIC == 'consensus_rank'
    edge_plot['_edge_raw'] = pd.to_numeric(edge_plot[EDGE_IMPORTANCE_METRIC], errors='coerce')
    edge_plot['_edge_score'] = -edge_plot['_edge_raw'] if edge_rank_like else edge_plot['_edge_raw']
    if EDGE_SCOPE == 'among_top_nodes':
        edge_plot = edge_plot[edge_plot['roi_src'].isin(shown_rois) & edge_plot['roi_dst'].isin(shown_rois)].copy()
    edge_plot = edge_plot.sort_values('_edge_score', ascending=False).head(TOP_K_3D_EDGES).copy()
    edge_plot['_edge_width'] = _norm(edge_plot['_edge_score'].abs().fillna(0), 1.2, 9.0)
    edge_plot['_src_cat'] = edge_plot['roi_src'].map(lambda r: _clean_category(node_lookup.get(int(r), {}).get('_category', 'Unknown')))
    edge_plot['_dst_cat'] = edge_plot['roi_dst'].map(lambda r: _clean_category(node_lookup.get(int(r), {}).get('_category', 'Unknown')))
    edge_plot['_edge_type'] = np.where(edge_plot['_src_cat'] == edge_plot['_dst_cat'], 'within-category', 'between-category')

traces, trace_filters = [], []
if SHOW_BRAIN_SURFACE:
    try:
        from nilearn import datasets, surface
        fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
        for hemi, surf_path in [('left', fsaverage.pial_left), ('right', fsaverage.pial_right)]:
            coords, faces = surface.load_surf_mesh(surf_path)
            traces.append(go.Mesh3d(
                x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color='#d8d2c4', opacity=BRAIN_SURFACE_OPACITY,
                name=f'{hemi} cortical surface', hoverinfo='skip', showscale=False,
                lighting=dict(ambient=0.55, diffuse=0.65, specular=0.12, roughness=0.8),
                lightposition=dict(x=0, y=-120, z=120), showlegend=True
            ))
            trace_filters.append({'kind': 'brain', 'cats': set(categories)})
    except Exception as e:
        print(f'Could not render cortical surface ({type(e).__name__}: {e}). Continuing with nodes/edges only.')
if SHOW_ALL_ROIS_FAINT:
    traces.append(go.Scatter3d(
        x=centroids[:, 0], y=centroids[:, 1], z=centroids[:, 2],
        mode='markers', marker=dict(size=3, color='lightgray', opacity=0.16),
        hoverinfo='skip', name='All Shen-268 ROIs', showlegend=True
    ))
    trace_filters.append({'kind': 'background', 'cats': set(categories)})

for cat in categories:
    sub = top_nodes[top_nodes['_category'] == cat].copy()
    idx = sub['roi_index'].astype(int).to_numpy()
    xyz = centroids[idx]
    hover = []
    for _, row in sub.iterrows():
        node_no = int(row['node_no']) if 'node_no' in row and pd.notna(row['node_no']) else int(row['roi_index']) + 1
        parts = [f'ROI {node_no}', f'Category: {row["_category"]}', f'roi_index={int(row["roi_index"])}']
        for c in ['roi_label', 'region_name', 'region_detail', 'hemisphere', 'lobe']:
            if c in sub.columns and pd.notna(row.get(c, '')) and str(row[c]).strip():
                parts.append(f'{c}: {row[c]}')
        parts.append(f'{NODE_IMPORTANCE_METRIC}: {float(row["_node_score_raw"]):.4g}')
        hover.append('<br>'.join(parts))
    traces.append(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='markers+text',
        text=[str(int(n)) for n in sub.get('node_no', sub['roi_index'] + 1)],
        textposition='top center', hovertext=hover, hoverinfo='text',
        marker=dict(
            size=sub['_node_size'], color=sub['_node_score'], cmin=cmin, cmax=cmax,
            colorscale=NODE_INTENSITY_COLORSCALE, colorbar=dict(title=NODE_IMPORTANCE_METRIC), showscale=len([t for t in trace_filters if t.get('kind') == 'node']) == 0,
            opacity=0.94, line=dict(width=1, color='black')
        ),
        name=f'Nodes: {cat}', showlegend=True
    ))
    trace_filters.append({'kind': 'node', 'cats': {cat}})

edge_type_seen = set()
for _, row in edge_plot.iterrows():
    s, d = int(row['roi_src']), int(row['roi_dst'])
    sx, sy, sz = centroids[s]
    dx, dy, dz = centroids[d]
    edge_type = row['_edge_type']
    color = '#2a78c4' if edge_type == 'within-category' else '#d4552d'
    src_no = int(node_lookup.get(s, {}).get('node_no', s + 1))
    dst_no = int(node_lookup.get(d, {}).get('node_no', d + 1))
    hover = '<br>'.join([
        f'ROI {src_no} -> ROI {dst_no}',
        f'Edge type: {edge_type}',
        f'Source category: {row["_src_cat"]}',
        f'Destination category: {row["_dst_cat"]}',
        f'{EDGE_IMPORTANCE_METRIC}: {float(row["_edge_raw"]):.4g}',
    ])
    traces.append(go.Scatter3d(
        x=[sx, dx], y=[sy, dy], z=[sz, dz], mode='lines',
        line=dict(color=color, width=float(row['_edge_width'])),
        opacity=0.72, hovertext=hover, hoverinfo='text',
        name=edge_type, legendgroup=edge_type, showlegend=edge_type not in edge_type_seen
    ))
    edge_type_seen.add(edge_type)
    trace_filters.append({'kind': 'edge', 'cats': {row['_src_cat'], row['_dst_cat']}})

fig = go.Figure(data=traces)
buttons = []
buttons.append(dict(label='All categories', method='update', args=[{'visible': [True] * len(traces)}]))
for cat in categories:
    visible = []
    for f in trace_filters:
        if f['kind'] == 'brain':
            visible.append(SHOW_BRAIN_SURFACE)
        elif f['kind'] == 'background':
            visible.append(SHOW_ALL_ROIS_FAINT)
        else:
            visible.append(cat in f['cats'])
    buttons.append(dict(label=cat, method='update', args=[{'visible': visible}]))
fig.update_layout(
    title=f'3D brain surface with Shen-268 importance graph: top {TOP_K_3D_NODES} nodes, top {len(edge_plot)} edges',
    scene=dict(
        xaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        yaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        zaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        bgcolor='rgba(0,0,0,0)',
        aspectmode='data',
        camera=dict(eye=dict(x=1.7, y=1.7, z=1.25)),
    ),
    updatemenus=[dict(buttons=buttons, direction='down', x=0.01, y=0.98, xanchor='left', yanchor='top')],
    legend=dict(x=0.01, y=0.82),
    margin=dict(l=0, r=0, t=58, b=0),
    height=820,
)
fig.show()

brain_dir = out_dir / 'brain_plots'
brain_dir.mkdir(parents=True, exist_ok=True)
html_path = brain_dir / f'interactive_3d_nodes_edges_top{TOP_K_3D_NODES}_{TOP_K_3D_EDGES}_{target_suffix}.html'
fig.write_html(str(html_path), include_plotlyjs='cdn')
print(f'Saved interactive 3D brain: {html_path}')


In [ ]:
# Interactive 3D brain: all 268 Shen ROIs colored by 8 brain categories
# This plot answers: where is each category/system in the brain?
CATEGORY_COL = 'network'
SHOW_CATEGORY_BRAIN_SURFACE = True
CATEGORY_BRAIN_SURFACE_OPACITY = 0.13
CATEGORY_NODE_SIZE = 6

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

if 'out_dir' not in globals():
    out_dir = Path(DRIVE_OUTPUT_DIR) if 'DRIVE_OUTPUT_DIR' in globals() else Path('/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability')

if 'target_suffix' not in globals():
    node_candidates = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)
    if not node_candidates:
        raise FileNotFoundError(f'No node_importance_*.csv found in {out_dir}')
    target_suffix = node_candidates[-1].stem.replace('node_importance_', '', 1)
    target_folds = [target_suffix.split('_+')[0]]
    print(f'Inferred target_suffix from saved CSV: {target_suffix}')

if 'agg' not in globals():
    node_csv = out_dir / f'node_importance_{target_suffix}.csv'
    if not node_csv.exists():
        node_csv = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)[-1]
    print(f'Loading node importance: {node_csv}')
    agg = pd.read_csv(node_csv)

if 'centroids' not in globals():
    if '_load_shen_coords' in globals():
        centroids, networks_csv, regions_csv = _load_shen_coords()
    else:
        coord_candidates = [
            Path(globals().get('ATLAS_CSV_PATH', '')),
            Path('/content/drive/MyDrive/GNN-mri/data/shen268_coords.csv'),
            Path('/content/drive/MyDrive/GNN-mri/data/shen_268_coords.csv'),
            Path('/content/GNN-mri/data/shen268_coords.csv'),
            Path('/content/GNN-mri/data/shen_268_coords.csv'),
        ]
        centroids = None
        for p in coord_candidates:
            if p and p.exists():
                dfc = pd.read_csv(p)
                cols = {str(c).lower().strip(): c for c in dfc.columns}
                x_col = next((cols[k] for k in ['x_mni', 'x', 'mni_x'] if k in cols), None)
                y_col = next((cols[k] for k in ['y_mni', 'y', 'mni_y'] if k in cols), None)
                z_col = next((cols[k] for k in ['z_mni', 'z', 'mni_z'] if k in cols), None)
                if x_col and y_col and z_col:
                    nodeno_col = next((cols[k] for k in ['nodeno', 'node_no', 'node'] if k in cols), None)
                    if nodeno_col:
                        dfc = dfc.sort_values(nodeno_col)
                    centroids = dfc[[x_col, y_col, z_col]].to_numpy(dtype=float)
                    print(f'Loaded MNI coordinates from {p}')
                    break
    if centroids is None:
        raise FileNotFoundError('Could not load Shen-268 MNI coordinates for 3D category plotting.')

def _clean_category(v):
    if pd.isna(v) or not str(v).strip() or str(v).lower() == 'nan':
        return 'Unknown'
    return str(v).strip()

cat_df = agg.copy()
cat_df['roi_index'] = cat_df['roi_index'].astype(int)
cat_df['_category'] = cat_df[CATEGORY_COL].map(_clean_category) if CATEGORY_COL in cat_df.columns else 'Unknown'
cat_df = cat_df.sort_values('roi_index').drop_duplicates('roi_index')

category_palette = {
    'Medial Frontal': '#1f77b4',
    'Frontoparietal': '#ff7f0e',
    'Default Mode': '#2ca02c',
    'Subcortical/Cerebellum': '#d62728',
    'Motor': '#9467bd',
    'Visual I': '#8c564b',
    'Visual II': '#e377c2',
    'Visual Association': '#17becf',
    'Unknown': '#7f7f7f',
}
fallback_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#17becf', '#7f7f7f']
categories = sorted(cat_df['_category'].unique().tolist())
for i, cat in enumerate(categories):
    category_palette.setdefault(cat, fallback_palette[i % len(fallback_palette)])

traces = []
trace_categories = []
if SHOW_CATEGORY_BRAIN_SURFACE:
    try:
        from nilearn import datasets, surface
        fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
        for hemi, surf_path in [('left', fsaverage.pial_left), ('right', fsaverage.pial_right)]:
            coords, faces = surface.load_surf_mesh(surf_path)
            traces.append(go.Mesh3d(
                x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color='#d8d2c4', opacity=CATEGORY_BRAIN_SURFACE_OPACITY,
                name=f'{hemi} cortical surface', hoverinfo='skip', showscale=False,
                lighting=dict(ambient=0.55, diffuse=0.65, specular=0.12, roughness=0.8),
                lightposition=dict(x=0, y=-120, z=120), showlegend=True
            ))
            trace_categories.append('brain')
    except Exception as e:
        print(f'Could not render cortical surface ({type(e).__name__}: {e}). Continuing with ROI categories only.')

label_col = 'roi_label' if 'roi_label' in cat_df.columns else ('region_name' if 'region_name' in cat_df.columns else None)
for cat in categories:
    sub = cat_df[cat_df['_category'] == cat].copy()
    idx = sub['roi_index'].astype(int).to_numpy()
    xyz = centroids[idx]
    hover = []
    for _, row in sub.iterrows():
        node_no = int(row['node_no']) if 'node_no' in row and pd.notna(row['node_no']) else int(row['roi_index']) + 1
        parts = [f'ROI {node_no}', f'Category: {cat}', f'roi_index={int(row["roi_index"])}']
        if label_col and pd.notna(row.get(label_col, '')) and str(row[label_col]).strip():
            parts.append(str(row[label_col]))
        for c in ['hemisphere', 'lobe', 'region_detail', 'consensus_z', 'consensus_rank']:
            if c in sub.columns and pd.notna(row.get(c, '')) and str(row[c]).strip():
                val = row[c]
                parts.append(f'{c}: {float(val):.4g}' if isinstance(val, (int, float, np.floating)) else f'{c}: {val}')
        hover.append('<br>'.join(parts))
    traces.append(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='markers', hovertext=hover, hoverinfo='text',
        marker=dict(size=CATEGORY_NODE_SIZE, color=category_palette[cat], opacity=0.94, line=dict(width=0.8, color='black')),
        name=f'{cat} ({len(sub)})', showlegend=True
    ))
    trace_categories.append(cat)

buttons = [dict(label='All categories', method='update', args=[{'visible': [True] * len(traces)}])]
for cat in categories:
    visible = [(tc == 'brain' and SHOW_CATEGORY_BRAIN_SURFACE) or (tc == cat) for tc in trace_categories]
    buttons.append(dict(label=cat, method='update', args=[{'visible': visible}]))

fig_cat = go.Figure(data=traces)
fig_cat.update_layout(
    title='3D Shen-268 brain categories: all 268 ROIs colored by brain system',
    scene=dict(
        xaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        yaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        zaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        bgcolor='rgba(0,0,0,0)',
        aspectmode='data', camera=dict(eye=dict(x=1.7, y=1.7, z=1.25)),
    ),
    updatemenus=[dict(buttons=buttons, direction='down', x=0.01, y=0.98, xanchor='left', yanchor='top')],
    legend=dict(title='Brain categories', x=0.01, y=0.82),
    margin=dict(l=0, r=0, t=58, b=0),
    height=820,
)
fig_cat.show()

cat_counts = cat_df['_category'].value_counts().rename_axis('category').reset_index(name='n_rois')
cat_counts['color'] = cat_counts['category'].map(category_palette)
print('Where is what: category color legend and ROI counts')
print(cat_counts.sort_values('category').to_string(index=False))

brain_dir = out_dir / 'brain_plots'
brain_dir.mkdir(parents=True, exist_ok=True)
html_path = brain_dir / f'interactive_3d_8categories_all268_{target_suffix}.html'
fig_cat.write_html(str(html_path), include_plotlyjs='cdn')
cat_counts.to_csv(brain_dir / f'category_legend_all268_{target_suffix}.csv', index=False)
print(f'Saved 8-category 3D brain: {html_path}')
print(f'Saved category legend CSV: {brain_dir / f"category_legend_all268_{target_suffix}.csv"}')


In [ ]:
# Interactive 3D brain with:
# - original node size: abs(consensus_z), normalized across all ROIs
# - original distinct node color: consensus_z, locally scaled across top nodes
# - hover shows both consensus_z and consensus_z_norm01
# - edges use edge_grad width and within/between color if network/category exists

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

TOP_K_3D_NODES = 15
TOP_K_3D_EDGES = 30

NODE_IMPORTANCE_METRIC = 'consensus_z'
EDGE_IMPORTANCE_METRIC = 'edge_grad'
NODE_CATEGORY_COL = 'network'
EDGE_SCOPE = 'among_top_nodes'  # 'among_top_nodes' or 'top_edges_overall'

NODE_INTENSITY_COLORSCALE = [
    [0.0, '#eeeeee'],
    [0.35, '#fff2a8'],
    [0.7, '#ffd84d'],
    [1.0, '#f2a900'],
]

SHOW_BRAIN_SURFACE = True
BRAIN_SURFACE_OPACITY = 0.16
SHOW_ALL_ROIS_FAINT = False

# Change paths if needed
node_csv = Path('/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability/node_importance_graphs_outer1_inner1_+4.csv')
edge_csv = Path('/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability/edge_importance_graphs_outer1_inner1_+4.csv')
coords_csv = Path('/content/drive/MyDrive/GNN-mri/data/shen268_coords.csv')

out_dir = node_csv.parent
target_suffix = node_csv.stem.replace('node_importance_', '', 1)

node_df = pd.read_csv(node_csv)
edge_df = pd.read_csv(edge_csv) if edge_csv.exists() else None
coords_df = pd.read_csv(coords_csv)

cols = {str(c).lower().strip(): c for c in coords_df.columns}
x_col = next((cols[k] for k in ['x_mni', 'x', 'mni_x'] if k in cols), None)
y_col = next((cols[k] for k in ['y_mni', 'y', 'mni_y'] if k in cols), None)
z_col = next((cols[k] for k in ['z_mni', 'z', 'mni_z'] if k in cols), None)
nodeno_col = next((cols[k] for k in ['nodeno', 'node_no', 'node'] if k in cols), None)

if not all([x_col, y_col, z_col]):
    raise ValueError(f'Could not find x/y/z coordinate columns in {coords_df.columns.tolist()}')

if nodeno_col:
    coords_df = coords_df.sort_values(nodeno_col)

centroids = coords_df[[x_col, y_col, z_col]].to_numpy(dtype=float)

def _norm(values, low, high):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or abs(mx - mn) < 1e-12:
        return np.full_like(arr, (low + high) / 2, dtype=float)
    return low + (arr - mn) * (high - low) / (mx - mn)

def _clean_category(v):
    if pd.isna(v) or not str(v).strip() or str(v).lower() == 'nan':
        return 'Unknown'
    return str(v).strip()

# -------------------------
# Nodes
# -------------------------
required_node_cols = {'roi_index', 'consensus_z'}
missing = required_node_cols - set(node_df.columns)
if missing:
    raise ValueError(f'Missing node columns: {missing}. Available: {node_df.columns.tolist()}')

node_df = node_df.copy()
node_df['roi_index'] = node_df['roi_index'].astype(int)
node_df['consensus_z'] = pd.to_numeric(node_df['consensus_z'], errors='coerce').fillna(0)

if 'consensus_z_norm01' not in node_df.columns:
    cz = node_df['consensus_z']
    if abs(cz.max() - cz.min()) < 1e-12:
        node_df['consensus_z_norm01'] = 0.5
    else:
        node_df['consensus_z_norm01'] = (cz - cz.min()) / (cz.max() - cz.min())
else:
    node_df['consensus_z_norm01'] = pd.to_numeric(
        node_df['consensus_z_norm01'],
        errors='coerce'
    ).fillna(0).clip(0, 1)

node_df['_node_score_raw'] = node_df['consensus_z']
node_df['_node_score'] = node_df['consensus_z']
node_df['_node_magnitude'] = node_df['_node_score'].abs()

# Original size logic: normalize abs(consensus_z) across ALL ROIs
node_df['_node_size'] = _norm(node_df['_node_magnitude'].fillna(0), 7, 24)

if NODE_CATEGORY_COL in node_df.columns:
    node_df['_category'] = node_df[NODE_CATEGORY_COL].map(_clean_category)
else:
    node_df['_category'] = 'Unknown'

# Original top-node selection: raw consensus_z
top_nodes = (
    node_df
    .sort_values('_node_score', ascending=False)
    .head(TOP_K_3D_NODES)
    .copy()
)

shown_rois = set(top_nodes['roi_index'].astype(int))
node_lookup = node_df.set_index('roi_index').to_dict('index')
categories = sorted(top_nodes['_category'].dropna().unique().tolist())

# Distinct visual color like original, but display the colorbar as 0-1.
# This is a local 0-1 scale across the displayed top nodes, not the global consensus_z_norm01.
color_min = float(top_nodes['_node_score'].min())
color_max = float(top_nodes['_node_score'].max())
if abs(color_max - color_min) < 1e-12:
    top_nodes['_node_color01_local'] = 0.5
else:
    top_nodes['_node_color01_local'] = (top_nodes['_node_score'] - color_min) / (color_max - color_min)
cmin, cmax = 0.0, 1.0

# -------------------------
# Edges
# -------------------------
edge_plot = pd.DataFrame()

if edge_df is not None and len(edge_df):
    required_edge_cols = {'roi_src', 'roi_dst', EDGE_IMPORTANCE_METRIC}
    missing = required_edge_cols - set(edge_df.columns)
    if missing:
        raise ValueError(f'Missing edge columns: {missing}. Available: {edge_df.columns.tolist()}')

    edge_plot = edge_df[['roi_src', 'roi_dst', EDGE_IMPORTANCE_METRIC]].copy()
    edge_plot['roi_src'] = edge_plot['roi_src'].astype(int)
    edge_plot['roi_dst'] = edge_plot['roi_dst'].astype(int)
    edge_plot['_edge_raw'] = pd.to_numeric(edge_plot[EDGE_IMPORTANCE_METRIC], errors='coerce').fillna(0)
    edge_plot['_edge_score'] = edge_plot['_edge_raw']

    if EDGE_SCOPE == 'among_top_nodes':
        edge_plot = edge_plot[
            edge_plot['roi_src'].isin(shown_rois) &
            edge_plot['roi_dst'].isin(shown_rois)
        ].copy()

    # If too few edges among top nodes, fall back to top edges overall
    if len(edge_plot) == 0:
        print('No edges among top nodes; falling back to top_edges_overall.')
        edge_plot = edge_df[['roi_src', 'roi_dst', EDGE_IMPORTANCE_METRIC]].copy()
        edge_plot['roi_src'] = edge_plot['roi_src'].astype(int)
        edge_plot['roi_dst'] = edge_plot['roi_dst'].astype(int)
        edge_plot['_edge_raw'] = pd.to_numeric(edge_plot[EDGE_IMPORTANCE_METRIC], errors='coerce').fillna(0)
        edge_plot['_edge_score'] = edge_plot['_edge_raw']

    edge_plot = edge_plot.sort_values('_edge_score', ascending=False).head(TOP_K_3D_EDGES).copy()
    edge_plot['_edge_width'] = _norm(edge_plot['_edge_score'].abs().fillna(0), 1.2, 9.0)

    edge_plot['_src_cat'] = edge_plot['roi_src'].map(
        lambda r: _clean_category(node_lookup.get(int(r), {}).get('_category', 'Unknown'))
    )
    edge_plot['_dst_cat'] = edge_plot['roi_dst'].map(
        lambda r: _clean_category(node_lookup.get(int(r), {}).get('_category', 'Unknown'))
    )
    edge_plot['_edge_type'] = np.where(
        edge_plot['_src_cat'] == edge_plot['_dst_cat'],
        'within-category',
        'between-category'
    )

# -------------------------
# Plot
# -------------------------
traces, trace_filters = [], []

if SHOW_BRAIN_SURFACE:
    try:
        from nilearn import datasets, surface

        fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
        for hemi, surf_path in [('left', fsaverage.pial_left), ('right', fsaverage.pial_right)]:
            coords, faces = surface.load_surf_mesh(surf_path)
            traces.append(go.Mesh3d(
                x=coords[:, 0],
                y=coords[:, 1],
                z=coords[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color='#d8d2c4',
                opacity=BRAIN_SURFACE_OPACITY,
                name=f'{hemi} cortical surface',
                hoverinfo='skip',
                showscale=False,
                lighting=dict(ambient=0.55, diffuse=0.65, specular=0.12, roughness=0.8),
                lightposition=dict(x=0, y=-120, z=120),
                showlegend=True,
            ))
            trace_filters.append({'kind': 'brain', 'cats': set(categories)})
    except Exception as e:
        print(f'Could not render cortical surface ({type(e).__name__}: {e}). Continuing with nodes/edges only.')

if SHOW_ALL_ROIS_FAINT:
    traces.append(go.Scatter3d(
        x=centroids[:, 0],
        y=centroids[:, 1],
        z=centroids[:, 2],
        mode='markers',
        marker=dict(size=3, color='lightgray', opacity=0.16),
        hoverinfo='skip',
        name='All Shen-268 ROIs',
        showlegend=True,
    ))
    trace_filters.append({'kind': 'background', 'cats': set(categories)})

for cat in categories:
    sub = top_nodes[top_nodes['_category'] == cat].copy()
    idx = sub['roi_index'].astype(int).to_numpy()
    xyz = centroids[idx]

    hover = []
    for _, row in sub.iterrows():
        node_no = int(row['node_no']) if 'node_no' in row and pd.notna(row['node_no']) else int(row['roi_index']) + 1
        parts = [
            f'ROI {node_no}',
            f'Category: {row["_category"]}',
            f'roi_index={int(row["roi_index"])}',
            f'consensus_z={float(row["consensus_z"]):.4f}',
            f'consensus_z_norm01={float(row["consensus_z_norm01"]):.4f}',
            f'local_color01={float(row["_node_color01_local"]):.4f}',
        ]

        for c in ['roi_label', 'region_name', 'region_detail', 'hemisphere', 'lobe']:
            if c in sub.columns and pd.notna(row.get(c, '')) and str(row[c]).strip():
                parts.append(f'{c}: {row[c]}')

        hover.append('<br>'.join(parts))

    traces.append(go.Scatter3d(
        x=xyz[:, 0],
        y=xyz[:, 1],
        z=xyz[:, 2],
        mode='markers+text',
        text=[str(int(n)) for n in sub.get('node_no', sub['roi_index'] + 1)],
        textposition='top center',
        hovertext=hover,
        hoverinfo='text',
        marker=dict(
            size=sub['_node_size'],
            color=sub['_node_color01_local'],
            cmin=cmin,
            cmax=cmax,
            colorscale=NODE_INTENSITY_COLORSCALE,
            colorbar=dict(title='local color<br>0-1'),
            showscale=len([t for t in trace_filters if t.get('kind') == 'node']) == 0,
            opacity=0.94,
            line=dict(width=1, color='black'),
        ),
        name=f'Nodes: {cat}',
        showlegend=True,
    ))
    trace_filters.append({'kind': 'node', 'cats': {cat}})

edge_type_seen = set()

for _, row in edge_plot.iterrows():
    s, d = int(row['roi_src']), int(row['roi_dst'])
    sx, sy, sz = centroids[s]
    dx, dy, dz = centroids[d]

    edge_type = row['_edge_type']
    color = '#2a78c4' if edge_type == 'within-category' else '#d4552d'

    src_no = int(node_lookup.get(s, {}).get('node_no', s + 1))
    dst_no = int(node_lookup.get(d, {}).get('node_no', d + 1))

    hover = '<br>'.join([
        f'ROI {src_no} -> ROI {dst_no}',
        f'Edge type: {edge_type}',
        f'Source category: {row["_src_cat"]}',
        f'Destination category: {row["_dst_cat"]}',
        f'{EDGE_IMPORTANCE_METRIC}: {float(row["_edge_raw"]):.6g}',
    ])

    traces.append(go.Scatter3d(
        x=[sx, dx],
        y=[sy, dy],
        z=[sz, dz],
        mode='lines',
        line=dict(color=color, width=float(row['_edge_width'])),
        opacity=0.72,
        hovertext=hover,
        hoverinfo='text',
        name=edge_type,
        legendgroup=edge_type,
        showlegend=edge_type not in edge_type_seen,
    ))

    edge_type_seen.add(edge_type)
    trace_filters.append({'kind': 'edge', 'cats': {row['_src_cat'], row['_dst_cat']}})

fig = go.Figure(data=traces)

buttons = []
buttons.append(dict(label='All categories', method='update', args=[{'visible': [True] * len(traces)}]))

for cat in categories:
    visible = []
    for f in trace_filters:
        if f['kind'] == 'brain':
            visible.append(SHOW_BRAIN_SURFACE)
        elif f['kind'] == 'background':
            visible.append(SHOW_ALL_ROIS_FAINT)
        else:
            visible.append(cat in f['cats'])
    buttons.append(dict(label=cat, method='update', args=[{'visible': visible}]))

fig.update_layout(
    title=f'3D brain surface with Shen-268 importance graph: top {TOP_K_3D_NODES} nodes, top {len(edge_plot)} edges',
    scene=dict(
        xaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        yaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        zaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=''),
        bgcolor='rgba(0,0,0,0)',
        aspectmode='data',
        camera=dict(eye=dict(x=1.7, y=1.7, z=1.25)),
    ),
    updatemenus=[dict(buttons=buttons, direction='down', x=0.01, y=0.98, xanchor='left', yanchor='top')],
    legend=dict(x=0.01, y=0.82),
    margin=dict(l=0, r=0, t=58, b=0),
    height=820,
)

fig.show()

brain_dir = out_dir / 'brain_plots'
brain_dir.mkdir(parents=True, exist_ok=True)

html_path = brain_dir / f'interactive_3d_nodes_edges_top{TOP_K_3D_NODES}_{TOP_K_3D_EDGES}_{target_suffix}.html'
fig.write_html(str(html_path), include_plotlyjs='cdn')

print(f'Saved interactive 3D brain: {html_path}')
print(f'Plotted {len(top_nodes)} nodes and {len(edge_plot)} edges.')


In [ ]:
# Static brain plots matching the interactive 3D version
# - minimal node columns: roi_index, consensus_z, consensus_z_norm01
# - minimal edge columns: roi_src, roi_dst, edge_grad
# - node size: abs(consensus_z), normalized across all ROIs, same as 3D
# - node colour: local 0-1 rescale of top-node consensus_z, same visual contrast as 3D
# - edge width/selection: edge_grad
# - edge colour: within-category blue, between-category orange when network/category is available

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex
from nilearn import plotting

TOP_K_STATIC_NODES = 15
TOP_K_STATIC_EDGES = 30
EDGE_SCOPE = 'among_top_nodes'  # 'among_top_nodes' or 'top_edges_overall'
NODE_CATEGORY_COL = 'network'

NODE_INTENSITY_COLORS = ['#eeeeee', '#fff2a8', '#ffd84d', '#f2a900']
NODE_CMAP = LinearSegmentedColormap.from_list('fbnetgen_node_gold', NODE_INTENSITY_COLORS)
WITHIN_EDGE_COLOR = '#2a78c4'
BETWEEN_EDGE_COLOR = '#d4552d'

if 'out_dir' not in globals():
    out_dir = Path(DRIVE_OUTPUT_DIR) if 'DRIVE_OUTPUT_DIR' in globals() else Path('/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability')

if 'target_suffix' not in globals():
    node_candidates = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)
    if not node_candidates:
        raise FileNotFoundError(f'No node_importance_*.csv found in {out_dir}')
    target_suffix = node_candidates[-1].stem.replace('node_importance_', '', 1)
    print(f'Inferred target_suffix from saved CSV: {target_suffix}')

node_csv = out_dir / f'node_importance_{target_suffix}.csv'
edge_csv = out_dir / f'edge_importance_{target_suffix}.csv'
if not node_csv.exists():
    node_csv = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)[-1]
if not edge_csv.exists():
    edge_csv = sorted(out_dir.glob('edge_importance_*.csv'), key=lambda p: p.stat().st_mtime)[-1]

print(f'Loading nodes: {node_csv}')
print(f'Loading edges: {edge_csv}')

node_df = pd.read_csv(node_csv)
edge_df = pd.read_csv(edge_csv, usecols=lambda c: c in ['roi_src', 'roi_dst', 'edge_grad'])

needed_node_cols = {'roi_index', 'consensus_z'}
missing = needed_node_cols - set(node_df.columns)
if missing:
    raise ValueError(f'Missing required node columns: {sorted(missing)}')
if 'consensus_z_norm01' not in node_df.columns:
    cz = pd.to_numeric(node_df['consensus_z'], errors='coerce')
    mn, mx = float(cz.min()), float(cz.max())
    node_df['consensus_z_norm01'] = 0.5 if abs(mx - mn) < 1e-12 else (cz - mn) / (mx - mn)

if 'centroids' not in globals():
    if '_load_shen_coords' in globals():
        centroids, networks_csv, regions_csv = _load_shen_coords()
    else:
        coord_candidates = [
            Path(globals().get('ATLAS_CSV_PATH', '')),
            Path('/content/drive/MyDrive/GNN-mri/data/shen268_coords.csv'),
            Path('/content/drive/MyDrive/GNN-mri/data/shen_268_coords.csv'),
            Path('/content/GNN-mri/data/shen268_coords.csv'),
            Path('/content/GNN-mri/data/shen_268_coords.csv'),
        ]
        centroids = None
        for p in coord_candidates:
            if p and p.exists():
                coords_df = pd.read_csv(p)
                cols = {str(c).lower().strip(): c for c in coords_df.columns}
                x_col = next((cols[k] for k in ['x_mni', 'x', 'mni_x'] if k in cols), None)
                y_col = next((cols[k] for k in ['y_mni', 'y', 'mni_y'] if k in cols), None)
                z_col = next((cols[k] for k in ['z_mni', 'z', 'mni_z'] if k in cols), None)
                node_no_col = next((cols[k] for k in ['nodeno', 'node_no', 'node'] if k in cols), None)
                if x_col and y_col and z_col:
                    if node_no_col:
                        coords_df = coords_df.sort_values(node_no_col)
                    centroids = coords_df[[x_col, y_col, z_col]].to_numpy(dtype=float)
                    print(f'Loaded MNI coordinates from {p}')
                    break
    if centroids is None:
        raise FileNotFoundError('Could not load Shen-268 MNI coordinates.')

node_df['roi_index'] = node_df['roi_index'].astype(int)
node_df['consensus_z'] = pd.to_numeric(node_df['consensus_z'], errors='coerce').fillna(0.0)
node_df['consensus_z_norm01'] = pd.to_numeric(node_df['consensus_z_norm01'], errors='coerce').fillna(0.5).clip(0, 1)

def _norm(values, low, high):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or abs(mx - mn) < 1e-12:
        return np.full_like(arr, (low + high) / 2, dtype=float)
    return low + (arr - mn) * (high - low) / (mx - mn)

node_df['_node_score'] = node_df['consensus_z']
node_df['_node_size'] = _norm(node_df['_node_score'].abs().fillna(0), 30, 220)

top_nodes = node_df.sort_values('_node_score', ascending=False).head(TOP_K_STATIC_NODES).copy()
shown_rois = set(top_nodes['roi_index'].astype(int))

color_min = float(top_nodes['_node_score'].min())
color_max = float(top_nodes['_node_score'].max())
if abs(color_max - color_min) < 1e-12:
    top_nodes['_node_color01_local'] = 0.5
else:
    top_nodes['_node_color01_local'] = (top_nodes['_node_score'] - color_min) / (color_max - color_min)
top_nodes['_marker_color'] = [to_hex(NODE_CMAP(float(v))) for v in top_nodes['_node_color01_local']]

node_lookup = node_df.set_index('roi_index').to_dict('index')

def _clean_category(v):
    if pd.isna(v) or not str(v).strip() or str(v).lower() == 'nan':
        return 'Unknown'
    return str(v).strip()

edge_df['roi_src'] = edge_df['roi_src'].astype(int)
edge_df['roi_dst'] = edge_df['roi_dst'].astype(int)
edge_df['edge_grad'] = pd.to_numeric(edge_df['edge_grad'], errors='coerce').fillna(0.0)
edge_plot = edge_df.copy()
if EDGE_SCOPE == 'among_top_nodes':
    edge_plot = edge_plot[edge_plot['roi_src'].isin(shown_rois) & edge_plot['roi_dst'].isin(shown_rois)].copy()
edge_plot = edge_plot.sort_values('edge_grad', ascending=False).head(TOP_K_STATIC_EDGES).copy()
edge_plot['_edge_width'] = _norm(edge_plot['edge_grad'].abs(), 1.2, 7.5)

has_category = NODE_CATEGORY_COL in node_df.columns
if has_category:
    edge_plot['_src_cat'] = edge_plot['roi_src'].map(lambda r: _clean_category(node_lookup.get(int(r), {}).get(NODE_CATEGORY_COL, 'Unknown')))
    edge_plot['_dst_cat'] = edge_plot['roi_dst'].map(lambda r: _clean_category(node_lookup.get(int(r), {}).get(NODE_CATEGORY_COL, 'Unknown')))
    edge_plot['_edge_type'] = np.where(edge_plot['_src_cat'] == edge_plot['_dst_cat'], 'within-category', 'between-category')
else:
    edge_plot['_edge_type'] = 'between-category'

print(f'  nodes shown: {len(top_nodes)}')
print(f'  edges shown: {len(edge_plot)}')
if len(edge_plot):
    print(f'  edge weight range: [{edge_plot.edge_grad.min():.6g}, {edge_plot.edge_grad.max():.6g}]')

brain_dir = out_dir / 'brain_plots'
brain_dir.mkdir(parents=True, exist_ok=True)

plot_specs = []
if has_category and (edge_plot['_edge_type'] == 'within-category').any():
    plot_specs.append(('within', edge_plot[edge_plot['_edge_type'] == 'within-category'], WITHIN_EDGE_COLOR))
if (edge_plot['_edge_type'] == 'between-category').any():
    plot_specs.append(('between', edge_plot[edge_plot['_edge_type'] == 'between-category'], BETWEEN_EDGE_COLOR))
plot_specs.append(('all_edges', edge_plot, BETWEEN_EDGE_COLOR))

for edge_name, sub_edges, edge_color in plot_specs:
    if len(sub_edges) == 0:
        continue

    adj = np.zeros((len(centroids), len(centroids)), dtype=float)
    for _, row in sub_edges.iterrows():
        s, d = int(row.roi_src), int(row.roi_dst)
        w = float(row.edge_grad)
        adj[s, d] = w
        adj[d, s] = w

    edge_cmap = LinearSegmentedColormap.from_list(f'{edge_name}_edge_cmap', [edge_color, edge_color])

    for mode, title in [('ortho', 'Front + side + top'), ('lyrz', '4-panel anatomical'), ('lzry', '4-panel anatomical alt')]:
        fig = plt.figure(figsize=(20, 7), facecolor='white')
        display = plotting.plot_connectome(
            adjacency_matrix=adj,
            node_coords=centroids,
            node_size=0,
            edge_threshold=None,
            edge_cmap=edge_cmap,
            edge_kwargs={'linewidth': 1.5, 'alpha': 0.72},
            display_mode=mode,
            colorbar=False,
            title=f'Top-{len(sub_edges)} {edge_name.replace("_", " ")} edges - {title}',
            figure=fig,
            axes=(0.02, 0.05, 0.86, 0.90),  # reserve blank space on the right for the node colorbar
        )

        display_coords = centroids[top_nodes['roi_index'].to_numpy()]
        display.add_markers(
            marker_coords=display_coords,
            marker_color=top_nodes['_marker_color'].tolist(),
            marker_size=top_nodes['_node_size'].to_numpy(),
        )

        sm = plt.cm.ScalarMappable(cmap=NODE_CMAP, norm=plt.Normalize(vmin=0, vmax=1))
        sm.set_array([])
        # Nilearn display axes are GlassBrainAxes, not normal Matplotlib axes.
        # Put the colorbar in the blank right margin reserved by axes= above.
        cax = fig.add_axes([0.915, 0.24, 0.014, 0.52])
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label('local color 0-1', labelpad=10)
        cbar.ax.tick_params(labelsize=9, pad=2)

        fp = brain_dir / f'static_3dmatch_{edge_name}_top{TOP_K_STATIC_NODES}_{len(sub_edges)}_{mode}_{target_suffix}.png'
        display.savefig(str(fp), dpi=180)
        print(f'Saved {edge_name} static brain ({mode}): {fp}')
        plt.show()
        plt.close('all')

print(f'Saved static 3D-matched brain plots to {brain_dir}')






In [ ]:
# Export clean website CSVs: node.csv + edge.csv
# node.csv columns:
#   node_no, x_mni, y_mni, z_mni, consensus_z, consensus_z_norm01, network, roi_label
# edge.csv columns:
#   roi_src, roi_dst, edge_grad, edge_type

from pathlib import Path
import numpy as np
import pandas as pd

if 'out_dir' not in globals():
    out_dir = Path(DRIVE_OUTPUT_DIR) if 'DRIVE_OUTPUT_DIR' in globals() else Path('/content/drive/MyDrive/results/fbnetgen_v5_full/interpretability')

if 'target_suffix' not in globals():
    node_candidates = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)
    if not node_candidates:
        raise FileNotFoundError(f'No node_importance_*.csv found in {out_dir}')
    target_suffix = node_candidates[-1].stem.replace('node_importance_', '', 1)
    print(f'Inferred target_suffix from saved CSV: {target_suffix}')

node_src_csv = out_dir / f'node_importance_{target_suffix}.csv'
edge_src_csv = out_dir / f'edge_importance_{target_suffix}.csv'
if not node_src_csv.exists():
    node_src_csv = sorted(out_dir.glob('node_importance_*.csv'), key=lambda p: p.stat().st_mtime)[-1]
if not edge_src_csv.exists():
    edge_src_csv = sorted(out_dir.glob('edge_importance_*.csv'), key=lambda p: p.stat().st_mtime)[-1]

print(f'Loading node source: {node_src_csv}')
print(f'Loading edge source: {edge_src_csv}')

node_raw = pd.read_csv(node_src_csv)
edge_raw = pd.read_csv(edge_src_csv)

# Load coordinates, preferring the notebook helper because it may already merge Shen labels/networks.
coord_df = None
if '_load_shen_coords' in globals():
    centroids, networks_csv, regions_csv = _load_shen_coords()
else:
    coord_candidates = [
        Path(globals().get('ATLAS_CSV_PATH', '')),
        Path('/content/drive/MyDrive/GNN-mri/data/shen268_coords.csv'),
        Path('/content/drive/MyDrive/GNN-mri/data/shen_268_coords.csv'),
        Path('/content/GNN-mri/data/shen268_coords.csv'),
        Path('/content/GNN-mri/data/shen_268_coords.csv'),
    ]
    centroids = None
    for p in coord_candidates:
        if p and p.exists():
            coord_df = pd.read_csv(p)
            cols = {str(c).lower().strip(): c for c in coord_df.columns}
            x_col = next((cols[k] for k in ['x_mni', 'x', 'mni_x'] if k in cols), None)
            y_col = next((cols[k] for k in ['y_mni', 'y', 'mni_y'] if k in cols), None)
            z_col = next((cols[k] for k in ['z_mni', 'z', 'mni_z'] if k in cols), None)
            node_no_col = next((cols[k] for k in ['nodeno', 'node_no', 'node'] if k in cols), None)
            if x_col and y_col and z_col:
                if node_no_col:
                    coord_df = coord_df.sort_values(node_no_col)
                centroids = coord_df[[x_col, y_col, z_col]].to_numpy(dtype=float)
                print(f'Loaded coordinates from {p}')
                break
    if centroids is None:
        raise FileNotFoundError('Could not load Shen-268 MNI coordinates.')

# Standardize node ids. roi_index is 0-based; node_no is 1-based for display/website labels.
node_raw['roi_index'] = node_raw['roi_index'].astype(int)
if 'node_no' not in node_raw.columns:
    node_raw['node_no'] = node_raw['roi_index'] + 1
node_raw['node_no'] = node_raw['node_no'].astype(int)

if 'consensus_z' not in node_raw.columns:
    raise ValueError('node source is missing consensus_z. Run the consensus_z cell first.')
node_raw['consensus_z'] = pd.to_numeric(node_raw['consensus_z'], errors='coerce').fillna(0.0)

if 'consensus_z_norm01' not in node_raw.columns:
    cz = node_raw['consensus_z']
    mn, mx = float(cz.min()), float(cz.max())
    node_raw['consensus_z_norm01'] = 0.5 if abs(mx - mn) < 1e-12 else (cz - mn) / (mx - mn)
node_raw['consensus_z_norm01'] = pd.to_numeric(node_raw['consensus_z_norm01'], errors='coerce').fillna(0.5).clip(0, 1)

# Attach coordinates by roi_index.
coords_by_roi = pd.DataFrame({
    'roi_index': np.arange(len(centroids), dtype=int),
    'x_mni': centroids[:, 0],
    'y_mni': centroids[:, 1],
    'z_mni': centroids[:, 2],
})
node_clean = node_raw.merge(coords_by_roi, on='roi_index', how='left', suffixes=('', '_coord'))

# Fill network/label from the Shen label table when the raw node CSV does not contain them.
if 'roi_label_df' not in globals() and 'build_shen268_label_table' in globals():
    roi_label_df = build_shen268_label_table(n_rois=len(centroids))

if 'roi_label_df' in globals():
    label_cols = [c for c in ['roi_index', 'network', 'roi_label'] if c in roi_label_df.columns]
    if len(label_cols) > 1:
        node_clean = node_clean.drop(columns=[c for c in ['network', 'roi_label'] if c in node_clean.columns])
        node_clean = node_clean.merge(roi_label_df[label_cols], on='roi_index', how='left')

# Keep network/label if available; otherwise create clean placeholders.
if 'network' not in node_clean.columns:
    node_clean['network'] = 'Unknown'
node_clean['network'] = node_clean['network'].fillna('Unknown').replace('', 'Unknown')

if 'roi_label' not in node_clean.columns:
    label_col = next((c for c in ['region_name', 'region_detail', 'label', 'name'] if c in node_clean.columns), None)
    node_clean['roi_label'] = node_clean[label_col].astype(str) if label_col else node_clean['node_no'].map(lambda n: f'ROI {int(n)}')
node_clean['roi_label'] = node_clean['roi_label'].fillna(node_clean['node_no'].map(lambda n: f'ROI {int(n)}'))

node_clean = node_clean[
    ['node_no', 'x_mni', 'y_mni', 'z_mni', 'consensus_z', 'consensus_z_norm01', 'network', 'roi_label']
].copy()
node_clean = node_clean.sort_values('node_no').reset_index(drop=True)

# Build a 0-based ROI -> network lookup from the cleaned node table for edge_type.
node_network_by_roi = node_clean.assign(roi_index=node_clean['node_no'].astype(int) - 1).set_index('roi_index')['network'].to_dict()

# Minimal edge CSV for website plotting.
needed_edge_cols = {'roi_src', 'roi_dst', 'edge_grad'}
missing_edges = needed_edge_cols - set(edge_raw.columns)
if missing_edges:
    raise ValueError(f'edge source is missing required columns: {sorted(missing_edges)}')

edge_clean = edge_raw[['roi_src', 'roi_dst', 'edge_grad']].copy()
edge_clean['roi_src'] = edge_clean['roi_src'].astype(int)
edge_clean['roi_dst'] = edge_clean['roi_dst'].astype(int)
edge_clean['edge_grad'] = pd.to_numeric(edge_clean['edge_grad'], errors='coerce').fillna(0.0)
edge_clean = edge_clean.sort_values('edge_grad', ascending=False).reset_index(drop=True)

# Add edge_type directly into edge.csv. It is within if source and destination share the same network.
edge_clean['src_network'] = edge_clean['roi_src'].map(node_network_by_roi).fillna('Unknown')
edge_clean['dst_network'] = edge_clean['roi_dst'].map(node_network_by_roi).fillna('Unknown')
edge_clean['edge_type'] = np.where(
    edge_clean['src_network'] == edge_clean['dst_network'],
    'within',
    'between',
)
edge_clean = edge_clean[['roi_src', 'roi_dst', 'edge_grad', 'edge_type']].copy()

website_dir = out_dir / 'website_csv'
website_dir.mkdir(parents=True, exist_ok=True)
node_out = website_dir / 'node.csv'
edge_out = website_dir / 'edge.csv'

node_clean.to_csv(node_out, index=False)
edge_clean.to_csv(edge_out, index=False)

print(f'Saved clean node CSV: {node_out}  rows={len(node_clean):,}')
print(f'Saved clean edge CSV: {edge_out}  rows={len(edge_clean):,}')
print('\nnode.csv columns:', node_clean.columns.tolist())
print('edge.csv columns:', edge_clean.columns.tolist())





In [ ]:
but

## How to read these brain plots

This section produced **two interactive 3D brains** plus several static views. Here is how to read them.

### What the model is doing

The model (FBNetGen) takes ~5–6 minutes of fMRI brain activity from a person, breaks it into short overlapping windows, builds one **functional connectivity graph per window** over **268 brain regions** (the Shen-268 atlas), and predicts a single behavioural score (here a cognitive score). We then ask the model two questions:

1. *"Which brain regions are you using to make that prediction?"*  →  **node importance**.
2. *"Which connections between regions are you using?"*  →  **edge importance**.

The plots below answer those two questions on top of an anatomical brain.

### Plot 1 — node markers (the "ROI plot")

* Every sphere is **one of 268 Shen-atlas regions**, placed at its real anatomical position in MNI space.
* **Colour** = importance score from the model (we used `consensus_z`, the mean of saliency, integrated gradients, occlusion, and pool-gate after each is z-scored). Bright yellow = highly important; dark red / black = much less important.
* **Size** = the same importance score (bigger sphere = more important).
* Hover any sphere to see `ROI <index> | <region name> | <network> | imp=<score>`.
* All 268 are shown, so the picture also conveys *coverage*: anatomical bias would show up as one hemisphere or one lobe being entirely yellow.

> Practical reading: yellow clusters = regions the model relies on most. If they coincide with known cognitive networks (e.g. default-mode, frontoparietal), that is corroboration. Diffuse yellow everywhere = the model uses the whole cortex roughly equally.

### Plot 2 — connectome (the "edges plot")

* The same 268 regions, but now the focus is **connections, not regions**.
* Only the **top-K edges** (default K = 50) are drawn — selected by `edge_grad`, the gradient of the prediction with respect to that edge's input weight. This is *"how much does the prediction change if I nudge this connection?"*.
* **Edge colour intensity** (Reds colormap) = edge importance — darker red = more important.
* **Spheres are sized** by the same node-importance score as Plot 1, but only nodes that *appear in at least one of the top-K edges* are visible. The rest are hidden so the figure is not cluttered.
* Hover any edge to see its endpoints; hover any sphere to see ROI label and importance.

> Practical reading: the connectome answers *"which pairs of regions are jointly informative?"* — a stronger claim than node importance, because it speaks to circuits rather than isolated nodes. A cluster of edges concentrated in one hemisphere or one lobe is a localised "circuit"; edges that span hemispheres or lobes suggest long-range integration.

### How node and edge plots relate

Node importance and edge importance can disagree, and that is informative:

* A node that is **important on its own** but appears in **few top edges** = the model uses that region's *intrinsic* feature (its time-series pattern) more than its connections.
* A node that is **mediocre on its own** but appears at the end of **many top edges** = the model uses that region as a *connectivity hub*.
* Regions in the **top of both** plots are the safest to highlight as "what the model relies on".

### What to look for / sanity check

* Are the high-importance regions clustered in known cognitive networks (DMN, frontoparietal, salience), or are they random? Clustering supports validity.
* Is there left/right symmetry? Most cognitive tasks show roughly bilateral patterns; gross asymmetry can be a fold artifact.
* Do the top edges span sensible distances? Real cognitive circuits tend to mix short-range (local processing) with long-range (integration) connections; a connectome that is *only* short or *only* long is suspicious.

### What the colours / scales actually mean

| Plot | Colour | Size |
|---|---|---|
| Markers (Plot 1) | `consensus_z` of node importance (saliency + integrated grad + occlusion + pool-gate, each z-scored, then averaged) | Same metric |
| Connectome (Plot 2) | `edge_grad` per edge — `\|∂ŷ/∂edge_weight\|` averaged over all test windows | Node sphere size still uses node importance |

Static PNG counterparts are saved in `brain_plots/` (front, side, top, ortho, 4-panel anatomical, 6-panel summary). Use them when you need a fixed-angle figure for a paper or slide; use the interactive HTML versions when you need to explore.

## How each importance score is computed (concepts)

Every score below is computed **per test window** (one functional graph from one subject) and then averaged across all windows of the chosen folds (~16,650 windows total for outer-1).

The scores answer subtly different questions, which is why the four node methods don't always agree — and that disagreement is itself useful information.

---

### Node importance — 4 methods, 4 different questions

#### 1. Pool-gate attention — *"How much weight did the model itself put on this region in the readout?"*

The model has a small built-in attention layer (the "pool gate") that, after the GNN finishes message passing, decides how much each of the 268 nodes contributes to the single graph-level vector that feeds the regression head. Those weights softmax to sum to 1 across the 268 nodes per graph.

We just **read those weights off** the model and average across windows. No perturbation, no gradient — it's the model's own self-reported attention.

**Caveat:** only sees the readout step. A node that mattered hugely earlier in the GNN can still get a low pool-gate weight.

#### 2. Saliency — *"Which regions, if their inputs changed slightly, would change the prediction the most?"*

Backpropagate the prediction back to each input feature, take the absolute value, and sum across the 268 features per node. Big number → small changes to that node's input would push the prediction.

It's a **local sensitivity** measure (one snapshot at the actual input). One backward pass per window — cheap.

**Caveat:** local. Saturated activations (ReLU/softmax) can give noisy or misleading gradients on a single sample.

#### 3. Integrated Gradients (IG) — *"As the input fades from nothing to its real value, how much did this node contribute to the resulting change in prediction?"*

Imagine starting from a "blank" input (all zeros) and gradually fading in the real connectivity matrix. At each step we compute the gradient and accumulate. Multiplied by the input difference, this gives a per-node attribution that:
- Is much smoother than vanilla saliency,
- Has the property that **summed across all features, it equals the change in prediction from baseline to real input**.

We approximate the integral with 32 small steps. ~32× more expensive than saliency.

#### 4. Occlusion — *"What if this region simply weren't there?"*

Run the model normally to get a baseline prediction. Then for each ROI in turn, **set its input feature row to zero**, re-run the model, and record how much the prediction shifted. Repeat for all 268 ROIs.

This is the most **causal-style** measure of the four — closest to a counterfactual experiment. It's also the slowest (one extra forward pass per ROI per batch).

**Caveat:** zeroing isn't always a "neutral" baseline; the model may not have seen all-zero rows during training, so very rare distributions can produce odd shifts.

---

### Combining them — `consensus_z`

The four methods are on completely different scales (pool-gate is in [0, 1]; saliency is in [0.4, 0.7]; IG is in [0.09, 0.28]; occlusion is in [0.004, 0.012]). So we can't just add them.

The fix is straightforward: **z-score each method across the 268 ROIs** (subtract its own mean, divide by its own std), then average the four z-scores per ROI:

```
consensus_z[i] = mean(z_pool[i], z_sal[i], z_IG[i], z_occ[i])
```

This is what colours the spheres in the brain plot. Higher z = ranked highly by multiple methods at once.

We also produce `consensus_rank`, which uses **rank averaging** instead — assign each ROI a rank 1..268 within each method, average the ranks, then re-sort. Rank averaging is less sensitive to outliers than z-score averaging; both are reported.

---

### Edge importance — 2 methods, 2 different questions

#### 5. GAT attention — *"Which edges did the message-passing layers attend to most?"*

Each GATv2 layer in the model has its own per-edge attention coefficient — basically the layer asking *"how much should ROI $i$ listen to ROI $j$ for this pass?"* . Your model has 3 GAT layers × 2 heads = 6 attention values per edge per window.

We **average across heads and layers**, then across windows. Same idea as pool-gate: read the model's own internal weights, no perturbation.

**Caveat:** in your run, GAT attention turned out nearly uniform (max/mean ≈ 1.14) — the model didn't strongly attend to any one edge. So this column is informative *as a finding* (the model uses connectivity broadly, not selectively) but not the most reliable for ranking.

#### 6. Edge gradient — *"If I nudge this connection's strength slightly, how much does the prediction move?"*

Backpropagate the prediction to the input edge weights and take the absolute value. Same local-sensitivity idea as node saliency, but applied to edges instead of node features.

This is the column the brain connectome uses to pick top-K edges by default — it has the most spread (max/mean ≈ 2.3) and is the easiest to interpret physically.

---

### Why four node methods instead of one?

Each method has a blind spot:

| | sees readout | sees GNN propagation | sees encoder | causal-ish | local-only |
|---|---|---|---|---|---|
| pool-gate  | yes | no  | no  | no   | n/a |
| saliency   | yes | yes | yes | no   | yes |
| IG         | yes | yes | yes | partial | no  |
| occlusion  | yes | yes | yes | yes  | no  |

A region with a high `consensus_z` is one that **multiple complementary methods all agree on** — much more defensible than any single method's top-30. Disagreements (one method ranks an ROI in the top 5, another ranks it in the bottom 30) are also informative: they often mean the model uses that region in one specific way (e.g. only via the readout, but not as a propagation hub).

---

### What the columns in the CSV / xlsx mean

| Column | Meaning |
|---|---|
| `<method>` (e.g. `pool_gate`) | raw average score across all windows × folds |
| `<method>_rank` | rank within all 268 ROIs (or all edges), 1 = most important |
| `consensus_rank` | mean of all `<method>_rank` columns, then re-sorted 1..N |
| `consensus_z` | mean z-scored value across methods (used for plot colours) |


In [ ]:
# Auto-summary printed alongside the plots (for new readers)
import numpy as np

print('=' * 70)
print('What you are looking at  (auto-summary from current run)')
print('=' * 70)

# Fold-level performance
test_r_mean = summary_df[summary_df.fold.isin(target_folds)]['test_subj_r'].mean()
test_r_std  = summary_df[summary_df.fold.isin(target_folds)]['test_subj_r'].std()
print(f'\nModel performance (subject-level Pearson r on held-out test):')
print(f'  Mean across {len(target_folds)} fold(s): {test_r_mean:.3f} +/- {test_r_std:.3f}')

# Node importance summary
print(f'\nNode importance (`consensus_z` of 4 attribution methods):')
print(f'  Range over 268 ROIs: [{node_imp.min():.3f}, {node_imp.max():.3f}]')
print(f'  Top-5 ROIs by importance:')
for r, idx in enumerate(np.argsort(node_imp)[::-1][:5]):
    line = f'    #{r+1}  ROI {idx:3d}  imp={node_imp[idx]:+.3f}'
    line += f'  MNI=({centroids[idx,0]:+5.1f}, {centroids[idx,1]:+5.1f}, {centroids[idx,2]:+5.1f})'
    if regions:  line += f'  {regions[idx]}'
    if networks: line += f'  [{networks[idx]}]'
    print(line)

# Edge importance summary
print(f'\nEdge importance (top-{TOP_EDGES} by `{EDGE_PLOT_METRIC}`):')
print(f'  Unique nodes touched: {len(set(top_e.roi_src.tolist() + top_e.roi_dst.tolist()))} / 268')
print(f'  Edge weight range:    [{adj[adj>0].min():.3g}, {adj.max():.3g}]')
print(f'  Top-5 connections:')
for r, (_, row) in enumerate(top_e.head(5).iterrows()):
    s, d = int(row.roi_src), int(row.roi_dst)
    extra_s = f' ({regions[s]})' if regions else ''
    extra_d = f' ({regions[d]})' if regions else ''
    print(f'    #{r+1}  ROI {s:3d}{extra_s}  <->  ROI {d:3d}{extra_d}    {EDGE_PLOT_METRIC}={row[EDGE_PLOT_METRIC]:.3g}')

print()
print('Reading guide:')
print('  - Yellow / large spheres in Plot 1  = regions the model relies on most.')
print('  - Dark-red lines in Plot 2          = the most informative connections.')
print('  - A region high in BOTH             = a robust finding.')
print('=' * 70)


## 15. Wrap-up

What gets written to the (versioned) output dir:

| File | What it is |
|---|---|
| `fbnetgen_interpretability_<fold>.xlsx` | **Main file** — multi-sheet Excel workbook (open this) |
| `fold_metric_summary.csv` | All folds' val/test r |
| `node_importance_<fold>.csv` | Per-ROI scores from all four methods + consensus rank |
| `edge_importance_<fold>.csv` | Per-edge `(roi_src, roi_dst)` GAT attention + edge gradient |
| `top_rois_bar.png`, `top_rois_heatmap.png` | Node visuals |
| `top_edges_bar.png`, `edge_attention_heatmap.png` | Edge visuals |
| **`brain_plots/`** | **Anatomical brain renders (nilearn)** — markers + connectome in multiple views, plus a 6-panel summary |

### Sheets in the xlsx

**Node-level**
* `top_30` / `node_importance` — top-K and full per-ROI importance.
* `fold_summary` — per-fold val/test r.
* `method_correlations` — Spearman ρ between the 4 node methods.
* `per_fold_long` — raw per-fold per-ROI scores.
* `config` — what settings produced this run.

**Edge-level**
* `top_30_edges` — top-K most important `(roi_src, roi_dst)` connections.
* `edge_importance` — full ranked edge table.
* `edge_hubs` — sum of incident edge importance per ROI (which nodes are connectivity hubs).

### Reading the columns

**Node:**
* `pool_gate` — what the model itself used to weight ROIs in the readout.
* `saliency` / `integrated_grad` — gradient-based attributions; reflect the full pipeline.
* `occlusion` — model-agnostic causal-style perturbation.
* `consensus_rank` — average rank across methods (1 = most important). ROIs ranked top-30 in 3+ methods are the safest bet.

**Edge:**
* `gat_attn` — model's own per-edge attention from GATv2Conv layers.
* `edge_grad` — `|∂ŷ/∂edge_attr|` — how sensitive the prediction is to each edge's LDW weight.
* `ldw_weight` — the original LDW connectivity strength (input feature, **not** an importance score). Useful to spot edges that are *both* strong in raw connectivity *and* used by the model.
* `consensus_rank` — average of `gat_attn` rank and `edge_grad` rank.

### Brain plot knobs (cells in section 12)

* `NODE_PLOT_METRIC` — which column of `agg` to colour nodes by. Default `consensus_z`. Try `pool_gate`, `saliency`, etc.
* `EDGE_PLOT_METRIC` — which column to size/colour edges by. Default `edge_grad`. Try `gat_attn`.
* `TOP_EDGES` — how many edges to draw. 30–50 reads cleanly; 100 looks like a network.